In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
InceptionV3 Liver Tumor Classification + XAI Pipeline (XAI Fixed)
=====================================================

Designed for Google Colab and based on:
Alfarhood et al. (2026), "Leveraging deep learning and explainable AI
for effective liver tumor classification from CT scan images."

Features
--------
1. Kaggle API authentication by uploading kaggle.json in Colab.
2. Automatic discovery of datasets attached to the cited Kaggle notebook:
   ahmedhamza1996/liver-tumor-classification
3. Optional direct Kaggle dataset slug.
4. Automatic image/class-folder discovery.
5. Stratified train/validation/test split.
6. Bilateral filtering + CLAHE + optional pseudocolor preprocessing.
7. On-the-fly augmentation.
8. ImageNet-pretrained InceptionV3 with two-stage fine-tuning.
9. Train, validation, and test metrics:
   accuracy, precision, recall/sensitivity, specificity, F1, AUC, loss.
10. Per-epoch train/validation/test curves.
11. Confusion matrices, ROC curves, box plots, classification reports.
12. Before/after preprocessing sample figures.
13. Model complexity:
    total/trainable/non-trainable parameters, FLOPs, GFLOPs, training time.
14. XAI for the first 20 test images:
    Grad-CAM, Grad-CAM++, LIME, SHAP, and Saliency maps.
15. Saves all outputs, creates ZIP, and automatically downloads it in Colab.

Important scientific note
-------------------------
The original paper combines a Kaggle source and Radiopaedia images.
This script downloads Kaggle input datasets attached to the cited Kaggle
notebook. Radiopaedia images are NOT automatically scraped. Add any
legally obtained external images to the dataset folder before execution.

Run in Colab
------------
    !python inceptionv3_liver_tumor_full_pipeline.py

The script will ask for kaggle.json when running in Colab.
"""

# ---------------------------------------------------------------------
# 0. INSTALL DEPENDENCIES
# ---------------------------------------------------------------------
import os
import sys
import json
import time
import math
import shutil
import random
import zipfile
import warnings
import subprocess
import importlib.util
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

def install_if_missing(import_name: str, pip_name: Optional[str] = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pkg in [
    ("kaggle", "kaggle"),
    ("cv2", "opencv-python-headless"),
    ("lime", "lime"),
    ("shap", "shap"),
    ("sklearn", "scikit-learn"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
]:
    install_if_missing(imp, pkg)

# ---------------------------------------------------------------------
# 1. IMPORTS
# ---------------------------------------------------------------------
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as inceptionv3_preprocess

from lime import lime_image
from skimage.segmentation import mark_boundaries
import shap

# ---------------------------------------------------------------------
# 2. CONFIGURATION
# ---------------------------------------------------------------------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 20
HEAD_EPOCHS = 20
FINE_TUNE_EPOCHS = 30
TOTAL_EPOCHS = HEAD_EPOCHS + FINE_TUNE_EPOCHS
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
TEST_SIZE = 0.20
VAL_SIZE_FROM_REMAINING = 0.20
N_XAI = 20

# Kaggle source cited in the base paper.
KAGGLE_KERNEL_REF = "ahmedhamza1996/liver-tumor-classification"

# Optional: provide an exact "owner/dataset-name" slug.
# Leave empty to discover datasets attached to KAGGLE_KERNEL_REF.
KAGGLE_DATASET_SLUG = ""

# Preprocessing from the base paper.
USE_BILATERAL_FILTER = True
USE_CLAHE = True
USE_PSEUDOCOLOR = True

# XAI runtime controls.
LIME_NUM_SAMPLES = 700
SHAP_BACKGROUND_SIZE = 12
XAI_BATCH_SIZE = 8

# Folders.
WORK_DIR = Path("/content/liver_inceptionv3_work") if Path("/content").exists() else Path.cwd() / "liver_inceptionv3_work"
DATA_DIR = WORK_DIR / "dataset"
RESULTS_DIR = WORK_DIR / "results_inceptionv3"
MODEL_DIR = RESULTS_DIR / "model"
PLOTS_DIR = RESULTS_DIR / "plots"
METRICS_DIR = RESULTS_DIR / "metrics"
XAI_DIR = RESULTS_DIR / "xai"
SAMPLES_DIR = RESULTS_DIR / "preprocessing_samples"

for p in [WORK_DIR, DATA_DIR, RESULTS_DIR, MODEL_DIR, PLOTS_DIR, METRICS_DIR, XAI_DIR, SAMPLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CLASS_ALIASES = {
    "normal": ["normal", "healthy", "no_tumor", "no-tumor", "notumor"],
    "benign": ["benign", "cyst", "hemangioma", "hydatid"],
    "malignant": ["malignant", "cancer", "hcc", "metastasis", "tumor"],
}
TARGET_CLASS_ORDER = ["normal", "benign", "malignant"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Bright colors requested for result curves.
BRIGHT_COLORS = {
    "train": "#00BFFF",
    "validation": "#FF1493",
    "test": "#32CD32",
    "loss_train": "#FF8C00",
    "loss_validation": "#9400D3",
    "loss_test": "#00CED1",
}

# ---------------------------------------------------------------------
# 3. REPRODUCIBILITY AND GPU
# ---------------------------------------------------------------------
def set_global_seed(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# Mixed precision accelerates training on modern Colab GPUs.
if gpus:
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision policy:", mixed_precision.global_policy())
    except Exception as exc:
        print("Mixed precision not enabled:", exc)

# ---------------------------------------------------------------------
# 4. KAGGLE AUTHENTICATION AND DATA DOWNLOAD
# ---------------------------------------------------------------------
def in_colab() -> bool:
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

def configure_kaggle_credentials() -> None:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"

    if target.exists():
        os.chmod(target, 0o600)
        print("Kaggle credentials found:", target)
        return

    local_candidates = [
        Path.cwd() / "kaggle.json",
        WORK_DIR / "kaggle.json",
        Path("/content/kaggle.json"),
    ]
    for candidate in local_candidates:
        if candidate.exists():
            shutil.copy2(candidate, target)
            os.chmod(target, 0o600)
            print("Configured Kaggle credentials from:", candidate)
            return

    if in_colab():
        from google.colab import files
        print("\nPlease upload kaggle.json...")
        uploaded = files.upload()
        json_files = [name for name in uploaded if name.lower().endswith(".json")]
        if not json_files:
            raise FileNotFoundError("No JSON file was uploaded.")
        source = Path(json_files[0])
        shutil.copy2(source, target)
        os.chmod(target, 0o600)
        print("Kaggle credentials configured.")
    else:
        raise FileNotFoundError(
            "kaggle.json was not found. Place it in the current directory "
            "or at ~/.kaggle/kaggle.json."
        )

def run_command(command: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    print("[CMD]", " ".join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

def parse_kernel_dataset_sources(metadata_dir: Path) -> List[str]:
    candidates = list(metadata_dir.rglob("*metadata*.json")) + list(metadata_dir.rglob("kernel-metadata.json"))
    slugs: List[str] = []
    for metadata_file in candidates:
        try:
            data = json.loads(metadata_file.read_text(encoding="utf-8"))
        except Exception:
            continue
        for key in ["dataset_sources", "datasetSources"]:
            values = data.get(key, [])
            if isinstance(values, list):
                for item in values:
                    if isinstance(item, str) and "/" in item:
                        slugs.append(item)
                    elif isinstance(item, dict):
                        ref = item.get("ref") or item.get("source") or item.get("dataset")
                        if isinstance(ref, str) and "/" in ref:
                            slugs.append(ref)
        # Newer metadata may use "data_sources".
        for item in data.get("data_sources", []) if isinstance(data.get("data_sources", []), list) else []:
            if isinstance(item, dict):
                ref = item.get("ref") or item.get("source")
                source_type = str(item.get("sourceType", item.get("type", ""))).lower()
                if isinstance(ref, str) and "/" in ref and ("dataset" in source_type or source_type == ""):
                    slugs.append(ref)
    return sorted(set(slugs))

def discover_kernel_inputs(kernel_ref: str) -> List[str]:
    metadata_dir = WORK_DIR / "kaggle_kernel_metadata"
    if metadata_dir.exists():
        shutil.rmtree(metadata_dir)
    metadata_dir.mkdir(parents=True, exist_ok=True)

    commands = [
        ["kaggle", "kernels", "pull", kernel_ref, "-p", str(metadata_dir), "-m"],
        ["kaggle", "kernels", "pull", "-p", str(metadata_dir), "-m", kernel_ref],
    ]
    output = ""
    success = False
    for cmd in commands:
        try:
            result = run_command(cmd, check=True)
            output += result.stdout or ""
            success = True
            break
        except Exception as exc:
            output += f"\n{exc}"
    if not success:
        print("Could not pull Kaggle notebook metadata.")
        print(output[-2000:])
        return []

    slugs = parse_kernel_dataset_sources(metadata_dir)
    print("Discovered Kaggle dataset sources:", slugs)
    return slugs

def download_kaggle_dataset(slug: str, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / f".downloaded_{slug.replace('/', '__')}"
    if marker.exists() and any(destination.rglob("*")):
        print("Dataset already downloaded:", slug)
        return
    result = run_command([
        "kaggle", "datasets", "download",
        "-d", slug,
        "-p", str(destination),
        "--unzip",
    ])
    print(result.stdout[-3000:])
    marker.touch()

def download_data() -> List[str]:
    configure_kaggle_credentials()
    dataset_slugs: List[str] = []

    if KAGGLE_DATASET_SLUG.strip():
        dataset_slugs = [KAGGLE_DATASET_SLUG.strip()]
    else:
        dataset_slugs = discover_kernel_inputs(KAGGLE_KERNEL_REF)

    if not dataset_slugs:
        raise RuntimeError(
            "\nNo attached Kaggle dataset slug could be detected automatically.\n"
            "Open the cited Kaggle notebook's Input tab, copy the dataset slug in\n"
            "'owner/dataset-name' format, and set KAGGLE_DATASET_SLUG near the\n"
            "top of this script.\n"
        )

    for slug in dataset_slugs:
        download_kaggle_dataset(slug, DATA_DIR / slug.replace("/", "__"))
    return dataset_slugs

# ---------------------------------------------------------------------
# 5. DATASET DISCOVERY
# ---------------------------------------------------------------------
def normalize_token(text: str) -> str:
    return text.lower().replace(" ", "_").replace("-", "_")

def infer_label_from_path(path: Path) -> Optional[str]:
    components = [normalize_token(part) for part in path.parts]
    # Prioritize exact directory-name matches.
    for label in TARGET_CLASS_ORDER:
        aliases = [normalize_token(x) for x in CLASS_ALIASES[label]]
        for component in reversed(components[:-1]):
            if component == label or component in aliases:
                return label
    # Then allow contained tokens, with "malignant" checked before generic tumor.
    joined = "/".join(components)
    for label in ["malignant", "benign", "normal"]:
        for alias in CLASS_ALIASES[label]:
            alias_n = normalize_token(alias)
            if alias_n in joined:
                return label
    return None

def is_valid_image(path: Path) -> Tuple[bool, str]:
    """Validate an image with OpenCV before it enters tf.data.

    Some Kaggle folders contain files with an image extension but corrupted,
    truncated, or non-image content. tf.io.decode_image raises an
    InvalidArgumentError for such files and stops training. OpenCV validation
    lets us skip them safely and write a diagnostic CSV.
    """
    try:
        raw = np.fromfile(str(path), dtype=np.uint8)
        if raw.size == 0:
            return False, "empty_file"
        image = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if image is None:
            return False, "opencv_decode_failed"
        if image.ndim != 3 or image.shape[0] < 8 or image.shape[1] < 8:
            return False, f"invalid_shape_{getattr(image, 'shape', None)}"
        return True, "ok"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"

def collect_images(root: Path) -> pd.DataFrame:
    rows = []
    bad_rows = []
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            label = infer_label_from_path(path)
            if label is None:
                continue
            valid, reason = is_valid_image(path)
            if valid:
                rows.append({"filepath": str(path), "label": label})
            else:
                bad_rows.append({"filepath": str(path), "label": label, "reason": reason})

    if bad_rows:
        bad_df = pd.DataFrame(bad_rows)
        bad_df.to_csv(METRICS_DIR / "skipped_corrupt_images.csv", index=False)
        print(f"Skipped {len(bad_df)} corrupted/unreadable image files.")
        print(bad_df.head(10).to_string(index=False))
    else:
        pd.DataFrame(columns=["filepath", "label", "reason"]).to_csv(
            METRICS_DIR / "skipped_corrupt_images.csv", index=False
        )

    df = pd.DataFrame(rows).drop_duplicates("filepath") if rows else pd.DataFrame(columns=["filepath", "label"])
    if df.empty:
        dirs = sorted({p.name for p in root.rglob("*") if p.is_dir()})
        print("Available directories (first 100):", dirs[:100])
        raise RuntimeError(
            "No valid images could be mapped to normal/benign/malignant classes. "
            "Update CLASS_ALIASES according to the downloaded folder names."
        )

    present = sorted(df["label"].unique().tolist())
    print("\nDetected classes:", present)
    print(df["label"].value_counts())
    if len(present) < 2:
        raise RuntimeError("At least two classes are required.")
    return df.reset_index(drop=True)

def create_splits(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    min_count = int(df["label"].value_counts().min())
    stratify_all = df["label"] if min_count >= 3 else None

    train_val, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=stratify_all,
    )
    min_remaining = int(train_val["label"].value_counts().min())
    stratify_remaining = train_val["label"] if min_remaining >= 3 else None

    train_df, val_df = train_test_split(
        train_val,
        test_size=VAL_SIZE_FROM_REMAINING,
        random_state=SEED,
        stratify=stratify_remaining,
    )

    for name, split in [("train", train_df), ("validation", val_df), ("test", test_df)]:
        split.to_csv(METRICS_DIR / f"{name}_split.csv", index=False)
        print(f"\n{name.upper()} ({len(split)} images)")
        print(split["label"].value_counts())

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

# ---------------------------------------------------------------------
# 6. PREPROCESSING
# ---------------------------------------------------------------------
def read_rgb_image(path: str) -> np.ndarray:
    image_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Unable to read image: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

def paper_preprocess_numpy(image_rgb: np.ndarray) -> np.ndarray:
    image_rgb = np.asarray(image_rgb, dtype=np.uint8)
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    if USE_BILATERAL_FILTER:
        gray = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

    if USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)

    if USE_PSEUDOCOLOR:
        color_bgr = cv2.applyColorMap(gray, cv2.COLORMAP_JET)
        processed = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2RGB)
    else:
        processed = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

    processed = cv2.resize(processed, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return processed.astype(np.float32)

def load_preprocess_from_path_numpy(path_value: Any) -> np.ndarray:
    """Decode by OpenCV from a path and apply the paper preprocessing.

    Using cv2.imdecode rather than tf.io.decode_image avoids TensorFlow PNG
    decoder crashes caused by malformed PNG metadata in a few Kaggle files.
    Files have already been validated in collect_images; this function retains
    a defensive zero-image fallback so one unexpected read error cannot stop
    an entire training run.
    """
    try:
        if isinstance(path_value, np.ndarray):
            path_value = path_value.item()
        if isinstance(path_value, (bytes, np.bytes_)):
            path_str = path_value.decode("utf-8")
        else:
            path_str = str(path_value)
        raw = np.fromfile(path_str, dtype=np.uint8)
        image_bgr = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise ValueError("OpenCV could not decode the image")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        processed = paper_preprocess_numpy(image_rgb)
        return inceptionv3_preprocess(processed.copy()).astype(np.float32)
    except Exception as exc:
        print(f"[WARNING] Runtime image read failed: {path_value!r} | {exc}")
        fallback = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        return inceptionv3_preprocess(fallback).astype(np.float32)

def load_and_preprocess(path: tf.Tensor, label: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    image = tf.numpy_function(load_preprocess_from_path_numpy, [path], tf.float32)
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    label = tf.cast(label, tf.int32)
    return image, label

def make_augmentation() -> keras.Sequential:
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical", seed=SEED),
            layers.RandomRotation(20.0 / 360.0, fill_mode="reflect", seed=SEED),
            layers.RandomTranslation(0.06, 0.06, fill_mode="reflect", seed=SEED),
            layers.RandomZoom(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10), seed=SEED),
            layers.RandomContrast(0.10, seed=SEED),
        ],
        name="training_augmentation",
    )

def build_dataset(
    df: pd.DataFrame,
    class_to_index: Dict[str, int],
    training: bool = False,
) -> tf.data.Dataset:
    paths = df["filepath"].astype(str).values
    labels = df["label"].map(class_to_index).astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=max(len(df), 1), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    # Additional protection against rare I/O failures in remote notebook runtimes.
    ds = ds.apply(tf.data.experimental.ignore_errors(log_warning=True))
    if training:
        augmenter = make_augmentation()
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def save_preprocessing_samples(df: pd.DataFrame, n: int = 9) -> None:
    sample_df = df.sample(min(n, len(df)), random_state=SEED)
    rows = len(sample_df)
    fig, axes = plt.subplots(rows, 2, figsize=(8, max(3 * rows, 6)))
    if rows == 1:
        axes = np.array([axes])
    for i, (_, row) in enumerate(sample_df.iterrows()):
        original = read_rgb_image(row["filepath"])
        processed = paper_preprocess_numpy(original).astype(np.uint8)
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f"Before: {row['label']}", fontsize=10, fontweight="bold")
        axes[i, 1].imshow(processed)
        axes[i, 1].set_title(f"After: {row['label']}", fontsize=10, fontweight="bold")
        axes[i, 0].axis("off")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.savefig(SAMPLES_DIR / "before_after_preprocessing_samples.png", dpi=300, bbox_inches="tight")
    plt.close()

# ---------------------------------------------------------------------
# 7. MODEL
# ---------------------------------------------------------------------
def build_inceptionv3(num_classes: int) -> Tuple[keras.Model, keras.Model]:
    """Build ImageNet-pretrained InceptionV3 for liver tumor classification."""
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="ct_image")
    backbone = InceptionV3(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling=None,
    )
    backbone.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dense(1024, activation="relu", name="dense_1024")(x)
    x = layers.BatchNormalization(name="head_batch_norm")(x)
    x = layers.Dropout(0.40, name="head_dropout")(x)
    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions"
    )(x)

    model = keras.Model(inputs, outputs, name="InceptionV3_Liver_Tumor")
    return model, backbone


def compile_model(model: keras.Model, learning_rate: float) -> None:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

def unfreeze_backbone(backbone: keras.Model, last_n_layers: int = 40) -> None:
    backbone.trainable = True
    for layer in backbone.layers[:-last_n_layers]:
        layer.trainable = False
    # Keep BatchNorm frozen for stable fine-tuning on a small medical dataset.
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

# ---------------------------------------------------------------------
# 8. METRICS
# ---------------------------------------------------------------------
def safe_multiclass_auc(y_true: np.ndarray, y_prob: np.ndarray, num_classes: int) -> float:
    try:
        if num_classes == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1]))
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")

def specificity_per_class(cm: np.ndarray) -> np.ndarray:
    total = cm.sum()
    values = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        values.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return np.asarray(values, dtype=float)

def per_class_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray, class_names: List[str]) -> pd.DataFrame:
    num_classes = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(num_classes), zero_division=0
    )
    specificity = specificity_per_class(cm)
    class_accuracy = []
    aucs = []
    y_bin = label_binarize(y_true, classes=np.arange(num_classes))
    if num_classes == 2 and y_bin.ndim == 2 and y_bin.shape[1] == 1:
        y_bin = np.concatenate([1 - y_bin, y_bin], axis=1)

    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        class_accuracy.append((tp + tn) / cm.sum() if cm.sum() else np.nan)
        try:
            aucs.append(roc_auc_score(y_bin[:, i], y_prob[:, i]))
        except Exception:
            aucs.append(np.nan)

    return pd.DataFrame({
        "class": class_names,
        "accuracy_ovr": class_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "auc_ovr": aucs,
        "support": support,
    })

def compute_split_metrics(
    model: keras.Model,
    ds: tf.data.Dataset,
    split_name: str,
    class_names: List[str],
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    eval_result = model.evaluate(ds, verbose=0, return_dict=True)
    y_true_parts, y_prob_parts = [], []
    for x_batch, y_batch in ds:
        probs = model.predict_on_batch(x_batch)
        y_true_parts.append(y_batch.numpy())
        y_prob_parts.append(np.asarray(probs))
    y_true = np.concatenate(y_true_parts)
    y_prob = np.concatenate(y_prob_parts)
    y_pred = np.argmax(y_prob, axis=1)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    specificity_macro = float(np.nanmean(specificity_per_class(cm)))
    auc_macro = safe_multiclass_auc(y_true, y_prob, len(class_names))

    metrics = {
        "split": split_name,
        "loss": float(eval_result["loss"]),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "sensitivity_macro": float(recall_macro),
        "specificity_macro": specificity_macro,
        "f1_macro": float(f1_macro),
        "auc_macro_ovr": auc_macro,
        "n_samples": int(len(y_true)),
    }

    class_df = per_class_metrics(y_true, y_pred, y_prob, class_names)
    class_df.insert(0, "split", split_name)

    pd.DataFrame([metrics]).to_csv(METRICS_DIR / f"{split_name}_overall_metrics.csv", index=False)
    class_df.to_csv(METRICS_DIR / f"{split_name}_per_class_metrics.csv", index=False)

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(METRICS_DIR / f"{split_name}_classification_report.csv")

    pred_df = pd.DataFrame({
        "true_index": y_true,
        "pred_index": y_pred,
        "true_class": [class_names[i] for i in y_true],
        "pred_class": [class_names[i] for i in y_pred],
        "predicted_confidence": np.max(y_prob, axis=1),
        "correct": (y_true == y_pred).astype(int),
    })
    for i, name in enumerate(class_names):
        pred_df[f"prob_{name}"] = y_prob[:, i]
    pred_df.to_csv(METRICS_DIR / f"{split_name}_predictions.csv", index=False)

    return metrics, y_true, y_pred, y_prob, class_df

# ---------------------------------------------------------------------
# 9. PER-EPOCH TEST CALLBACK
# ---------------------------------------------------------------------
class TestMetricsCallback(keras.callbacks.Callback):
    def __init__(self, test_ds: tf.data.Dataset, class_names: List[str]):
        super().__init__()
        self.test_ds = test_ds
        self.class_names = class_names
        self.records: List[Dict[str, float]] = []

    def on_epoch_end(self, epoch: int, logs: Optional[Dict[str, Any]] = None) -> None:
        logs = logs or {}
        result = self.model.evaluate(self.test_ds, verbose=0, return_dict=True)
        y_true_parts, y_prob_parts = [], []
        for xb, yb in self.test_ds:
            pb = self.model.predict_on_batch(xb)
            y_true_parts.append(yb.numpy())
            y_prob_parts.append(np.asarray(pb))
        y_true = np.concatenate(y_true_parts)
        y_prob = np.concatenate(y_prob_parts)
        y_pred = np.argmax(y_prob, axis=1)
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(self.class_names)))
        spec = float(np.nanmean(specificity_per_class(cm)))
        auc_value = safe_multiclass_auc(y_true, y_prob, len(self.class_names))
        record = {
            "epoch": epoch + 1,
            "test_loss": float(result["loss"]),
            "test_accuracy": float(accuracy_score(y_true, y_pred)),
            "test_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_sensitivity": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_specificity": spec,
            "test_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_auc": auc_value,
        }
        self.records.append(record)
        print(
            f" — test_acc: {record['test_accuracy']:.4f}"
            f" — test_prec: {record['test_precision']:.4f}"
            f" — test_rec/sens: {record['test_recall']:.4f}"
            f" — test_spec: {record['test_specificity']:.4f}"
            f" — test_auc: {record['test_auc']:.4f}"
        )

# ---------------------------------------------------------------------
# 10. PLOTS
# ---------------------------------------------------------------------
def save_confusion_matrix(cm: np.ndarray, class_names: List[str], split_name: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="turbo")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(class_names)), class_names, rotation=35, ha="right")
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("Predicted label", fontweight="bold")
    ax.set_ylabel("True label", fontweight="bold")
    ax.set_title(f"{split_name.title()} Confusion Matrix", fontweight="bold")
    threshold = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > threshold else "black", fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_roc_curves(y_true: np.ndarray, y_prob: np.ndarray, class_names: List[str], split_name: str) -> None:
    num_classes = len(class_names)
    y_bin = label_binarize(y_true, classes=np.arange(num_classes))
    if num_classes == 2 and y_bin.shape[1] == 1:
        y_bin = np.concatenate([1 - y_bin, y_bin], axis=1)

    fig, ax = plt.subplots(figsize=(8, 7))
    colors = plt.cm.hsv(np.linspace(0, 0.85, num_classes))
    for i, (name, color) in enumerate(zip(class_names, colors)):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, linewidth=2.5, color=color, label=f"{name} (AUC={roc_auc:.3f})")
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, color="black")
    ax.set_xlabel("False Positive Rate", fontweight="bold")
    ax.set_ylabel("True Positive Rate", fontweight="bold")
    ax.set_title(f"{split_name.title()} ROC–AUC Curves", fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_roc_auc.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_metric_boxplot(class_df: pd.DataFrame, split_name: str) -> None:
    columns = ["accuracy_ovr", "precision", "recall_sensitivity", "specificity", "f1_score", "auc_ovr"]
    labels = ["Accuracy", "Precision", "Recall/\nSensitivity", "Specificity", "F1", "AUC"]
    data = [class_df[col].dropna().values for col in columns]
    fig, ax = plt.subplots(figsize=(10, 6))
    box = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(box["boxes"])))
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Metric value", fontweight="bold")
    ax.set_title(f"{split_name.title()} Per-Class Metric Box Plot", fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_metrics_boxplot.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_confidence_boxplot(y_true: np.ndarray, y_prob: np.ndarray, class_names: List[str], split_name: str) -> None:
    confidences = np.max(y_prob, axis=1)
    data = [confidences[y_true == i] for i in range(len(class_names))]
    fig, ax = plt.subplots(figsize=(8, 6))
    box = ax.boxplot(data, labels=class_names, patch_artist=True, showmeans=True)
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(box["boxes"])))
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Predicted confidence", fontweight="bold")
    ax.set_title(f"{split_name.title()} Confidence Distribution", fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_confidence_boxplot.png", dpi=300, bbox_inches="tight")
    plt.close()

def merge_histories(history1: keras.callbacks.History, history2: keras.callbacks.History) -> Dict[str, List[float]]:
    merged: Dict[str, List[float]] = {}
    keys = set(history1.history.keys()) | set(history2.history.keys())
    for key in keys:
        merged[key] = list(history1.history.get(key, [])) + list(history2.history.get(key, []))
    return merged

def save_training_curves(history: Dict[str, List[float]], test_records: List[Dict[str, float]]) -> None:
    epochs = np.arange(1, len(history.get("accuracy", [])) + 1)
    test_df = pd.DataFrame(test_records)
    history_df = pd.DataFrame(history)
    history_df.insert(0, "epoch", epochs)
    history_df.to_csv(METRICS_DIR / "training_history.csv", index=False)
    test_df.to_csv(METRICS_DIR / "per_epoch_test_metrics.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, history["accuracy"], linewidth=2.5, color=BRIGHT_COLORS["train"], label="Training")
    ax.plot(epochs, history["val_accuracy"], linewidth=2.5, color=BRIGHT_COLORS["validation"], label="Validation")
    if not test_df.empty:
        ax.plot(test_df["epoch"], test_df["test_accuracy"], linewidth=2.5,
                color=BRIGHT_COLORS["test"], label="Test")
    ax.axvline(HEAD_EPOCHS, linestyle="--", color="black", alpha=0.7, label="Fine-tuning starts")
    ax.set_xlabel("Epoch", fontweight="bold")
    ax.set_ylabel("Accuracy", fontweight="bold")
    ax.set_title("InceptionV3 Accuracy Curves", fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "accuracy_curves_train_validation_test.png", dpi=300, bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, history["loss"], linewidth=2.5, color=BRIGHT_COLORS["loss_train"], label="Training")
    ax.plot(epochs, history["val_loss"], linewidth=2.5, color=BRIGHT_COLORS["loss_validation"], label="Validation")
    if not test_df.empty:
        ax.plot(test_df["epoch"], test_df["test_loss"], linewidth=2.5,
                color=BRIGHT_COLORS["loss_test"], label="Test")
    ax.axvline(HEAD_EPOCHS, linestyle="--", color="black", alpha=0.7, label="Fine-tuning starts")
    ax.set_xlabel("Epoch", fontweight="bold")
    ax.set_ylabel("Loss", fontweight="bold")
    ax.set_title("InceptionV3 Loss Curves", fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "loss_curves_train_validation_test.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Test metric curves.
    if not test_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        metric_cols = [
            ("test_accuracy", "Accuracy"),
            ("test_precision", "Precision"),
            ("test_recall", "Recall/Sensitivity"),
            ("test_specificity", "Specificity"),
            ("test_auc", "AUC"),
        ]
        colors = plt.cm.hsv(np.linspace(0, 0.85, len(metric_cols)))
        for (col, label), color in zip(metric_cols, colors):
            ax.plot(test_df["epoch"], test_df[col], linewidth=2.2, label=label, color=color)
        ax.set_xlabel("Epoch", fontweight="bold")
        ax.set_ylabel("Metric value", fontweight="bold")
        ax.set_ylim(0, 1.05)
        ax.set_title("Per-Epoch Test Metrics", fontweight="bold")
        ax.legend(ncol=2)
        ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "per_epoch_test_metric_curves.png", dpi=300, bbox_inches="tight")
        plt.close()

# ---------------------------------------------------------------------
# 11. MODEL COMPLEXITY
# ---------------------------------------------------------------------
def count_parameters(model: keras.Model) -> Dict[str, int]:
    total = int(model.count_params())
    trainable = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    non_trainable = int(sum(np.prod(v.shape) for v in model.non_trainable_weights))
    return {
        "total_parameters": total,
        "trainable_parameters": trainable,
        "non_trainable_parameters": non_trainable,
    }

def estimate_flops(model: keras.Model) -> Tuple[Optional[int], Optional[float]]:
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        concrete = tf.function(lambda x: model(x, training=False)).get_concrete_function(
            tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32)
        )
        frozen = convert_variables_to_constants_v2(concrete)
        graph_def = frozen.graph.as_graph_def()
        with tf.Graph().as_default() as graph:
            tf.compat.v1.graph_util.import_graph_def(graph_def, name="")
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            profile = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd="op", options=opts)
            flops = int(profile.total_float_ops) if profile is not None else None
        return flops, (flops / 1e9 if flops is not None else None)
    except Exception as exc:
        print("FLOPs estimation failed:", exc)
        return None, None

# ---------------------------------------------------------------------
# 12. XAI HELPERS
# ---------------------------------------------------------------------
def last_conv_layer_name(model: keras.Model) -> str:
    """Return a suitable final InceptionV3 feature layer for CAM methods."""
    for candidate in ["mixed10", "mixed9_1", "mixed9", "mixed8"]:
        try:
            layer = model.get_layer(candidate)
            if len(layer.output.shape) == 4:
                return candidate
        except Exception:
            pass

    for layer in reversed(model.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except Exception:
            continue

    raise ValueError("No suitable 4-D InceptionV3 feature layer found.")


def normalize_heatmap(heatmap: np.ndarray) -> np.ndarray:
    """Return a safe, finite, contiguous float32 2-D heatmap in [0, 1].

    Mixed precision can produce float16 heatmaps, while OpenCV resize does not
    reliably support float16. This helper also handles singleton dimensions,
    NaN/Inf values, and degenerate all-zero maps.
    """
    heatmap = np.asarray(heatmap)
    heatmap = np.squeeze(heatmap)

    if heatmap.ndim == 3:
        # Collapse an unexpected channel dimension safely.
        heatmap = np.mean(np.abs(heatmap.astype(np.float32)), axis=-1)
    if heatmap.ndim != 2:
        raise ValueError(f"Expected a 2-D heatmap, received shape {heatmap.shape}")

    heatmap = np.ascontiguousarray(heatmap, dtype=np.float32)
    heatmap = np.nan_to_num(heatmap, nan=0.0, posinf=0.0, neginf=0.0)
    heatmap = np.maximum(heatmap, 0.0)
    maximum = float(np.max(heatmap)) if heatmap.size else 0.0
    if maximum <= 1e-12:
        return np.zeros_like(heatmap, dtype=np.float32)
    return np.ascontiguousarray(heatmap / maximum, dtype=np.float32)

def gradcam(model: keras.Model, image_batch: tf.Tensor, class_index: int, layer_name: str) -> np.ndarray:
    """Generate Grad-CAM without breaking the gradient path under mixed precision.

    Important: gradients must be requested with respect to the original tensor
    returned by the model. Casting that tensor inside GradientTape and then asking
    for gradients with respect to the cast tensor disconnects the graph and returns
    None. We therefore differentiate first and cast only afterwards.
    """
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output],
        name="gradcam_model",
    )

    image_batch = tf.cast(image_batch, model.inputs[0].dtype)
    with tf.GradientTape() as tape:
        raw_conv_output, raw_predictions = grad_model(image_batch, training=False)
        # Watch explicitly for compatibility with loaded Keras 3 models.
        tape.watch(raw_conv_output)
        predictions_f32 = tf.cast(raw_predictions, tf.float32)
        target_score = predictions_f32[:, int(class_index)]

    grads = tape.gradient(target_score, raw_conv_output)
    if grads is None:
        raise RuntimeError(
            f"Grad-CAM gradients are None for layer '{layer_name}'. "
            "Use a convolutional layer connected to the classifier output."
        )

    conv_output = tf.cast(raw_conv_output, tf.float32)
    grads = tf.cast(grads, tf.float32)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv_output, axis=-1)[0]
    return normalize_heatmap(cam.numpy())


def gradcam_plus_plus(model: keras.Model, image_batch: tf.Tensor, class_index: int, layer_name: str) -> np.ndarray:
    """Generate a numerically stable Grad-CAM++ map under mixed precision."""
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output],
        name="gradcam_plus_plus_model",
    )

    image_batch = tf.cast(image_batch, model.inputs[0].dtype)
    with tf.GradientTape() as tape:
        raw_conv_output, raw_predictions = grad_model(image_batch, training=False)
        tape.watch(raw_conv_output)
        predictions_f32 = tf.cast(raw_predictions, tf.float32)
        score = predictions_f32[:, int(class_index)]

    raw_grads = tape.gradient(score, raw_conv_output)
    if raw_grads is None:
        raise RuntimeError(
            f"Grad-CAM++ gradients are None for layer '{layer_name}'. "
            "Use a convolutional layer connected to the classifier output."
        )

    conv = tf.cast(raw_conv_output[0], tf.float32)
    grads = tf.cast(raw_grads[0], tf.float32)

    grads2 = tf.square(grads)
    grads3 = grads2 * grads
    spatial_sum = tf.reduce_sum(conv, axis=(0, 1), keepdims=True)
    denominator = 2.0 * grads2 + spatial_sum * grads3
    denominator = tf.where(
        tf.abs(denominator) > 1e-10,
        denominator,
        tf.ones_like(denominator),
    )
    alphas = grads2 / denominator
    alphas = tf.where(tf.math.is_finite(alphas), alphas, tf.zeros_like(alphas))

    positive_grads = tf.nn.relu(grads)
    weights = tf.reduce_sum(alphas * positive_grads, axis=(0, 1))
    cam = tf.reduce_sum(conv * weights[None, None, :], axis=-1)
    return normalize_heatmap(cam.numpy())

def saliency_map(model: keras.Model, image_batch: tf.Tensor, class_index: int) -> np.ndarray:
    image_var = tf.Variable(tf.cast(image_batch, tf.float32))
    with tf.GradientTape() as tape:
        predictions = tf.cast(model(image_var, training=False), tf.float32)
        score = predictions[:, class_index]
    gradients = tape.gradient(score, image_var)
    if gradients is None:
        raise RuntimeError("Saliency gradients are None.")
    saliency = tf.reduce_max(tf.abs(tf.cast(gradients[0], tf.float32)), axis=-1).numpy()
    return normalize_heatmap(saliency)

def heatmap_overlay(display_rgb: np.ndarray, heatmap: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Resize and overlay a heatmap without OpenCV float16 failures."""
    display_rgb = np.asarray(display_rgb)
    if display_rgb.ndim != 3 or display_rgb.shape[-1] != 3:
        raise ValueError(f"Expected RGB image with shape (H,W,3), got {display_rgb.shape}")
    display_rgb = np.ascontiguousarray(np.clip(display_rgb, 0, 255), dtype=np.uint8)

    safe_heatmap = normalize_heatmap(heatmap)
    target_size = (int(display_rgb.shape[1]), int(display_rgb.shape[0]))
    resized = cv2.resize(
        np.ascontiguousarray(safe_heatmap, dtype=np.float32),
        target_size,
        interpolation=cv2.INTER_CUBIC,
    )
    resized = np.nan_to_num(resized, nan=0.0, posinf=1.0, neginf=0.0)
    resized = np.clip(resized, 0.0, 1.0)
    heatmap_uint8 = np.ascontiguousarray(np.rint(resized * 255.0), dtype=np.uint8)

    colored_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(display_rgb, 1.0 - float(alpha), colored_rgb, float(alpha), 0.0)
    return np.ascontiguousarray(overlay, dtype=np.uint8)

def model_ready_from_rgb(rgb_uint8: np.ndarray) -> np.ndarray:
    processed = paper_preprocess_numpy(rgb_uint8)
    return inceptionv3_preprocess(processed.copy()).astype(np.float32)

def lime_explanation(
    model: keras.Model,
    display_rgb: np.ndarray,
    class_index: int,
) -> np.ndarray:
    explainer = lime_image.LimeImageExplainer(random_state=SEED)

    def predict_fn(images: np.ndarray) -> np.ndarray:
        batch = np.stack([model_ready_from_rgb(np.asarray(img, dtype=np.uint8)) for img in images])
        return model.predict(batch, batch_size=XAI_BATCH_SIZE, verbose=0)

    explanation = explainer.explain_instance(
        display_rgb.astype(np.double),
        classifier_fn=predict_fn,
        top_labels=max(3, class_index + 1),
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES,
        random_seed=SEED,
    )
    temp, mask = explanation.get_image_and_mask(
        class_index,
        positive_only=True,
        num_features=10,
        hide_rest=False,
    )
    temp = np.clip(temp, 0, 255).astype(np.uint8)
    boundary = mark_boundaries(temp / 255.0, mask)
    return np.uint8(np.clip(boundary, 0, 1) * 255)

def prepare_shap_explainer(model: keras.Model, background_batch: np.ndarray):
    try:
        return shap.GradientExplainer(model, background_batch)
    except Exception as exc:
        print("GradientExplainer failed; using DeepExplainer:", exc)
        return shap.DeepExplainer(model, background_batch)

def extract_shap_map(shap_values: Any, class_index: int) -> np.ndarray:
    # SHAP versions return either list[num_classes] of (B,H,W,C),
    # or one array (B,H,W,C,num_classes).
    if isinstance(shap_values, list):
        arr = np.asarray(shap_values[min(class_index, len(shap_values) - 1)])
        sample = arr[0]
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 5:
            sample = arr[0, ..., min(class_index, arr.shape[-1] - 1)]
        elif arr.ndim == 4:
            sample = arr[0]
        else:
            sample = np.squeeze(arr)
    if sample.ndim == 3:
        sample = np.mean(np.abs(sample), axis=-1)
    return normalize_heatmap(np.asarray(sample, dtype=np.float32))

def save_xai_for_first_test_images(
    model: keras.Model,
    test_df: pd.DataFrame,
    class_to_index: Dict[str, int],
    class_names: List[str],
    n_images: int = N_XAI,
) -> None:
    if test_df.empty:
        return

    technique_dirs = {}
    for technique in ["gradcam", "gradcam_plus_plus", "lime", "shap", "saliency", "combined"]:
        technique_dirs[technique] = XAI_DIR / technique
        technique_dirs[technique].mkdir(parents=True, exist_ok=True)

    layer_name = last_conv_layer_name(model)
    print("XAI convolutional layer:", layer_name)

    # SHAP background from the first available test images.
    bg_images = []
    for path in test_df["filepath"].head(SHAP_BACKGROUND_SIZE):
        rgb = read_rgb_image(path)
        bg_images.append(model_ready_from_rgb(rgb))
    background = np.stack(bg_images).astype(np.float32)
    shap_explainer = prepare_shap_explainer(model, background)

    xai_rows = []
    for number, (_, row) in enumerate(test_df.head(n_images).iterrows(), start=1):
        print(f"Generating XAI {number}/{min(n_images, len(test_df))}: {row['filepath']}")
        original = read_rgb_image(row["filepath"])
        display_rgb = cv2.resize(original, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        model_input = model_ready_from_rgb(original)
        batch = tf.convert_to_tensor(model_input[None, ...], dtype=tf.float32)
        probs = model.predict(batch, verbose=0)[0]
        pred_index = int(np.argmax(probs))
        true_index = class_to_index[row["label"]]

        # Grad-CAM.
        gc_heat = gradcam(model, batch, pred_index, layer_name)
        gc_img = heatmap_overlay(display_rgb, gc_heat)

        # Grad-CAM++.
        try:
            gcpp_heat = gradcam_plus_plus(model, batch, pred_index, layer_name)
            gcpp_img = heatmap_overlay(display_rgb, gcpp_heat)
        except Exception as exc:
            print("Grad-CAM++ fallback to Grad-CAM:", exc)
            gcpp_heat = gc_heat
            gcpp_img = gc_img

        # Saliency.
        sal_heat = saliency_map(model, batch, pred_index)
        sal_img = heatmap_overlay(display_rgb, sal_heat)

        # LIME.
        try:
            lime_img = lime_explanation(model, display_rgb, pred_index)
        except Exception as exc:
            print("LIME failed:", exc)
            lime_img = display_rgb.copy()

        # SHAP.
        try:
            shap_values = shap_explainer.shap_values(model_input[None, ...])
            shap_heat = extract_shap_map(shap_values, pred_index)
            shap_img = heatmap_overlay(display_rgb, shap_heat)
        except Exception as exc:
            print("SHAP failed:", exc)
            shap_img = display_rgb.copy()

        stem = f"Patient_{number:02d}_true_{class_names[true_index]}_pred_{class_names[pred_index]}"
        for technique, image in [
            ("gradcam", gc_img),
            ("gradcam_plus_plus", gcpp_img),
            ("lime", lime_img),
            ("shap", shap_img),
            ("saliency", sal_img),
        ]:
            cv2.imwrite(
                str(technique_dirs[technique] / f"{stem}.png"),
                cv2.cvtColor(image.astype(np.uint8), cv2.COLOR_RGB2BGR),
            )

        # Combined six-panel image.
        fig, axes = plt.subplots(2, 3, figsize=(13, 9))
        images = [display_rgb, gc_img, gcpp_img, lime_img, shap_img, sal_img]
        titles = [
            "Original",
            "Grad-CAM",
            "Grad-CAM++",
            "LIME",
            "SHAP",
            "Saliency Map",
        ]
        for ax, image, title in zip(axes.ravel(), images, titles):
            ax.imshow(image)
            ax.set_title(title, fontweight="bold")
            ax.axis("off")
        fig.suptitle(
            f"Patient {number:02d} | True: {class_names[true_index]} | "
            f"Predicted: {class_names[pred_index]} ({probs[pred_index]:.4f})",
            fontsize=13,
            fontweight="bold",
        )
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        combined_path = technique_dirs["combined"] / f"{stem}.png"
        plt.savefig(combined_path, dpi=250, bbox_inches="tight")
        plt.close()

        record = {
            "patient_number": number,
            "filepath": row["filepath"],
            "true_class": class_names[true_index],
            "predicted_class": class_names[pred_index],
            "confidence": float(probs[pred_index]),
            "correct": int(true_index == pred_index),
        }
        for idx, class_name in enumerate(class_names):
            record[f"prob_{class_name}"] = float(probs[idx])
        xai_rows.append(record)

        # Avoid graph accumulation during 20 repeated explanation runs.
        tf.keras.backend.clear_session if False else None

    pd.DataFrame(xai_rows).to_csv(XAI_DIR / "first_20_test_xai_predictions.csv", index=False)

# ---------------------------------------------------------------------
# 13. ZIP AND DOWNLOAD
# ---------------------------------------------------------------------
def zip_results() -> Path:
    zip_path = WORK_DIR / "InceptionV3_Liver_Tumor_Complete_Results.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in RESULTS_DIR.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=file_path.relative_to(RESULTS_DIR.parent))
    return zip_path

def auto_download(path: Path) -> None:
    if in_colab():
        from google.colab import files
        print("Starting automatic download:", path)
        files.download(str(path))
    else:
        print("Results ZIP:", path.resolve())

# ---------------------------------------------------------------------
# 14. MAIN
# ---------------------------------------------------------------------
def main() -> None:
    start_total = time.perf_counter()

    # A. Download.
    slugs = download_data()
    (RESULTS_DIR / "downloaded_kaggle_sources.txt").write_text("\n".join(slugs), encoding="utf-8")

    # B. Discover and split.
    all_df = collect_images(DATA_DIR)
    class_names = [name for name in TARGET_CLASS_ORDER if name in all_df["label"].unique()]
    # Include unexpected mapped labels safely.
    class_names += sorted(set(all_df["label"].unique()) - set(class_names))
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    (RESULTS_DIR / "class_mapping.json").write_text(
        json.dumps(class_to_index, indent=2), encoding="utf-8"
    )

    train_df, val_df, test_df = create_splits(all_df)
    save_preprocessing_samples(train_df, n=9)

    train_ds = build_dataset(train_df, class_to_index, training=True)
    train_eval_ds = build_dataset(train_df, class_to_index, training=False)
    val_ds = build_dataset(val_df, class_to_index, training=False)
    test_ds = build_dataset(test_df, class_to_index, training=False)

    # C. Build and profile before training.
    model, backbone = build_inceptionv3(len(class_names))
    compile_model(model, INITIAL_LR)
    print(f"Model: {model.name}")
    print(f"Total parameters: {model.count_params():,}")
    print(f"Trainable parameters (stage 1): {sum(int(np.prod(v.shape)) for v in model.trainable_weights):,}")
    print(f"Non-trainable parameters (stage 1): {sum(int(np.prod(v.shape)) for v in model.non_trainable_weights):,}")
    with open(MODEL_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
        model.summary(print_fn=lambda line: f.write(line + "\n"))

    complexity = count_parameters(model)
    flops, gflops = estimate_flops(model)
    complexity["flops_per_forward_pass"] = flops
    complexity["gflops_per_forward_pass"] = gflops

    # D. Train frozen head.
    best_model_path = MODEL_DIR / "best_inceptionv3_liver.keras"
    callbacks_common = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=4,
            min_lr=1e-7,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(METRICS_DIR / "training_log.csv"), append=False),
    ]

    test_callback_stage1 = TestMetricsCallback(test_ds, class_names)
    training_start = time.perf_counter()
    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks_common + [test_callback_stage1],
        verbose=1,
    )

    # E. Fine-tune.
    unfreeze_backbone(backbone, last_n_layers=50)
    compile_model(model, FINE_TUNE_LR)

    callbacks_finetune = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=4,
            min_lr=1e-8,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(METRICS_DIR / "fine_tuning_log.csv"), append=False),
    ]
    test_callback_stage2 = TestMetricsCallback(test_ds, class_names)
    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=HEAD_EPOCHS,
        epochs=TOTAL_EPOCHS,
        callbacks=callbacks_finetune + [test_callback_stage2],
        verbose=1,
    )
    training_time_seconds = time.perf_counter() - training_start

    # Load best model.
    if best_model_path.exists():
        model = keras.models.load_model(best_model_path)

    merged_history = merge_histories(history1, history2)
    test_records = test_callback_stage1.records + test_callback_stage2.records
    save_training_curves(merged_history, test_records)

    # F. Final split evaluations.
    all_metrics = []
    eval_payload = {}
    for split_name, ds in [
        ("training", train_eval_ds),
        ("validation", val_ds),
        ("test", test_ds),
    ]:
        metrics, y_true, y_pred, y_prob, class_df = compute_split_metrics(
            model, ds, split_name, class_names
        )
        all_metrics.append(metrics)
        eval_payload[split_name] = (y_true, y_pred, y_prob, class_df)
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
        save_confusion_matrix(cm, class_names, split_name)
        save_roc_curves(y_true, y_prob, class_names, split_name)
        save_metric_boxplot(class_df, split_name)
        save_confidence_boxplot(y_true, y_prob, class_names, split_name)

    overall_df = pd.DataFrame(all_metrics)
    overall_df.to_csv(METRICS_DIR / "all_split_overall_metrics.csv", index=False)
    print("\nFINAL METRICS\n", overall_df.to_string(index=False))

    # G. Complexity and timing.
    complexity.update({
        "training_time_seconds": float(training_time_seconds),
        "training_time_minutes": float(training_time_seconds / 60.0),
        "total_pipeline_time_seconds_before_xai": float(time.perf_counter() - start_total),
        "input_shape": [1, IMG_SIZE, IMG_SIZE, 3],
        "batch_size": BATCH_SIZE,
        "head_epochs_requested": HEAD_EPOCHS,
        "fine_tune_epochs_requested": FINE_TUNE_EPOCHS,
    })
    with open(METRICS_DIR / "model_complexity_and_time.json", "w", encoding="utf-8") as f:
        json.dump(complexity, f, indent=2)
    pd.DataFrame([complexity]).to_csv(METRICS_DIR / "model_complexity_and_time.csv", index=False)

    # H. Save final model.
    model.save(MODEL_DIR / "final_inceptionv3_liver.keras")

    # I. XAI first 20 test images.
    save_xai_for_first_test_images(
        model=model,
        test_df=test_df,
        class_to_index=class_to_index,
        class_names=class_names,
        n_images=N_XAI,
    )

    # J. Final metadata and ZIP.
    final_metadata = {
        "base_model": "InceptionV3",
        "class_names": class_names,
        "class_mapping": class_to_index,
        "kaggle_sources": slugs,
        "dataset_images_total": int(len(all_df)),
        "training_images": int(len(train_df)),
        "validation_images": int(len(val_df)),
        "test_images": int(len(test_df)),
        "preprocessing": {
            "bilateral_filter": USE_BILATERAL_FILTER,
            "clahe": USE_CLAHE,
            "pseudocolor": USE_PSEUDOCOLOR,
            "resize": [IMG_SIZE, IMG_SIZE],
        },
        "augmentation": {
            "rotation_degrees": 20,
            "translation_fraction": 0.06,
            "horizontal_vertical_flip": True,
            "zoom": 0.10,
            "contrast": 0.10,
        },
        "xai": ["Grad-CAM", "Grad-CAM++", "LIME", "SHAP", "Saliency Map"],
    }
    with open(RESULTS_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(final_metadata, f, indent=2)

    zip_path = zip_results()
    print("\nCompleted successfully.")
    print("Results directory:", RESULTS_DIR)
    print("ZIP file:", zip_path)
    auto_download(zip_path)

if __name__ == "__main__":
    main()


[INSTALL] lime
TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">

Please upload kaggle.json...


Saving kaggle.json to kaggle.json
Kaggle credentials configured.
[CMD] kaggle kernels pull ahmedhamza1996/liver-tumor-classification -p /content/liver_inceptionv3_work/kaggle_kernel_metadata -m
Discovered Kaggle dataset sources: ['ahmedhamza1996/dataset']
[CMD] kaggle datasets download -d ahmedhamza1996/dataset -p /content/liver_inceptionv3_work/dataset/ahmedhamza1996__dataset --unzip
Dataset URL: https://www.kaggle.com/datasets/ahmedhamza1996/dataset
License(s): unknown

  0%|          | 0.00/241M [00:00<?, ?B/s]
  1%|          | 3.00M/241M [00:00<00:47, 5.27MB/s]
  3%|▎         | 7.00M/241M [00:00<00:24, 9.95MB/s]
  7%|▋         | 17.0M/241M [00:01<00:15, 15.2MB/s]
 16%|█▌        | 39.0M/241M [00:01<00:07, 28.6MB/s]
 25%|██▌       | 61.0M/241M [00:02<00:03, 47.1MB/s]
 30%|███       | 73.0M/241M [00:02<00:03, 46.3MB/s]
 33%|███▎      | 79.0M/241M [00:02<00:03, 43.1MB/s]
 42%|████▏     | 101M/241M [00:02<00:02, 59.6MB/s] 
 45%|████▍     | 108M/241M [00:02<00:02, 56.7MB/s]
 54%|█████▎  

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model: InceptionV3_Liver_Tumor
Total parameters: 23,908,131
Trainable parameters (stage 1): 2,103,299
Non-trainable parameters (stage 1): 21,804,832


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


...

Generating XAI 20/20: /content/liver_inceptionv3_work/dataset/ahmedhamza1996__dataset/output/malignant/(55).PNG


  0%|          | 0/700 [00:00<?, ?it/s]


Completed successfully.
Results directory: /content/liver_inceptionv3_work/results_inceptionv3
ZIP file: /content/liver_inceptionv3_work/InceptionV3_Liver_Tumor_Complete_Results.zip
Starting automatic download: /content/liver_inceptionv3_work/InceptionV3_Liver_Tumor_Complete_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
from google.colab import files

zip_path = "/content/liver_inceptionv3_work/InceptionV3_Liver_Tumor_Complete_Results.zip"

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

6.Efficient V2S

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EfficientNetV2-S Liver Tumor Classification + XAI Pipeline
=====================================================

Designed for Google Colab and based on:
Alfarhood et al. (2026), "Leveraging deep learning and explainable AI
for effective liver tumor classification from CT scan images."

Features
--------
1. Kaggle API authentication by uploading kaggle.json in Colab.
2. Automatic discovery of datasets attached to the cited Kaggle notebook:
   ahmedhamza1996/liver-tumor-classification
3. Optional direct Kaggle dataset slug.
4. Automatic image/class-folder discovery.
5. Stratified train/validation/test split.
6. Bilateral filtering + CLAHE + optional pseudocolor preprocessing.
7. On-the-fly augmentation.
8. ImageNet-pretrained EfficientNetV2-S with two-stage fine-tuning.
9. Train, validation, and test metrics:
   accuracy, precision, recall/sensitivity, specificity, F1, AUC, loss.
10. Per-epoch train/validation/test curves.
11. Confusion matrices, ROC curves, box plots, classification reports.
12. Before/after preprocessing sample figures.
13. Model complexity:
    total/trainable/non-trainable parameters, FLOPs, GFLOPs, training time.
14. XAI for the first 20 test images:
    Grad-CAM, Grad-CAM++, LIME, SHAP, and Saliency maps.
15. Saves all outputs, creates ZIP, and automatically downloads it in Colab.

Important scientific note
-------------------------
The original paper combines a Kaggle source and Radiopaedia images.
This script downloads Kaggle input datasets attached to the cited Kaggle
notebook. Radiopaedia images are NOT automatically scraped. Add any
legally obtained external images to the dataset folder before execution.

Run in Colab
------------
    !python efficientnetv2s_liver_tumor_full_pipeline.py

The script will ask for kaggle.json when running in Colab.
"""

# ---------------------------------------------------------------------
# 0. INSTALL DEPENDENCIES
# ---------------------------------------------------------------------
import os
import sys
import json
import time
import math
import shutil
import random
import zipfile
import warnings
import subprocess
import importlib.util
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

def install_if_missing(import_name: str, pip_name: Optional[str] = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pkg in [
    ("kaggle", "kaggle"),
    ("cv2", "opencv-python-headless"),
    ("lime", "lime"),
    ("shap", "shap"),
    ("sklearn", "scikit-learn"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
]:
    install_if_missing(imp, pkg)

# ---------------------------------------------------------------------
# 1. IMPORTS
# ---------------------------------------------------------------------
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as efficientnetv2_preprocess

from lime import lime_image
from skimage.segmentation import mark_boundaries
import shap

# ---------------------------------------------------------------------
# 2. CONFIGURATION
# ---------------------------------------------------------------------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 20
HEAD_EPOCHS = 20
FINE_TUNE_EPOCHS = 30
TOTAL_EPOCHS = HEAD_EPOCHS + FINE_TUNE_EPOCHS
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
TEST_SIZE = 0.20
VAL_SIZE_FROM_REMAINING = 0.20
N_XAI = 20

# Kaggle source cited in the base paper.
KAGGLE_KERNEL_REF = "ahmedhamza1996/liver-tumor-classification"

# Optional: provide an exact "owner/dataset-name" slug.
# Leave empty to discover datasets attached to KAGGLE_KERNEL_REF.
KAGGLE_DATASET_SLUG = ""

# Preprocessing from the base paper.
USE_BILATERAL_FILTER = True
USE_CLAHE = True
USE_PSEUDOCOLOR = True

# XAI runtime controls.
LIME_NUM_SAMPLES = 700
SHAP_BACKGROUND_SIZE = 12
XAI_BATCH_SIZE = 8

# Folders.
WORK_DIR = Path("/content/liver_efficientnetv2s_work") if Path("/content").exists() else Path.cwd() / "liver_efficientnetv2s_work"
DATA_DIR = WORK_DIR / "dataset"
RESULTS_DIR = WORK_DIR / "results_efficientnetv2s"
MODEL_DIR = RESULTS_DIR / "model"
PLOTS_DIR = RESULTS_DIR / "plots"
METRICS_DIR = RESULTS_DIR / "metrics"
XAI_DIR = RESULTS_DIR / "xai"
SAMPLES_DIR = RESULTS_DIR / "preprocessing_samples"

for p in [WORK_DIR, DATA_DIR, RESULTS_DIR, MODEL_DIR, PLOTS_DIR, METRICS_DIR, XAI_DIR, SAMPLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CLASS_ALIASES = {
    "normal": ["normal", "healthy", "no_tumor", "no-tumor", "notumor"],
    "benign": ["benign", "cyst", "hemangioma", "hydatid"],
    "malignant": ["malignant", "cancer", "hcc", "metastasis", "tumor"],
}
TARGET_CLASS_ORDER = ["normal", "benign", "malignant"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Bright colors requested for result curves.
BRIGHT_COLORS = {
    "train": "#00BFFF",
    "validation": "#FF1493",
    "test": "#32CD32",
    "loss_train": "#FF8C00",
    "loss_validation": "#9400D3",
    "loss_test": "#00CED1",
}

# ---------------------------------------------------------------------
# 3. REPRODUCIBILITY AND GPU
# ---------------------------------------------------------------------
def set_global_seed(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# Mixed precision accelerates training on modern Colab GPUs.
if gpus:
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision policy:", mixed_precision.global_policy())
    except Exception as exc:
        print("Mixed precision not enabled:", exc)

# ---------------------------------------------------------------------
# 4. KAGGLE AUTHENTICATION AND DATA DOWNLOAD
# ---------------------------------------------------------------------
def in_colab() -> bool:
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

def configure_kaggle_credentials() -> None:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"

    if target.exists():
        os.chmod(target, 0o600)
        print("Kaggle credentials found:", target)
        return

    local_candidates = [
        Path.cwd() / "kaggle.json",
        WORK_DIR / "kaggle.json",
        Path("/content/kaggle.json"),
    ]
    for candidate in local_candidates:
        if candidate.exists():
            shutil.copy2(candidate, target)
            os.chmod(target, 0o600)
            print("Configured Kaggle credentials from:", candidate)
            return

    if in_colab():
        from google.colab import files
        print("\nPlease upload kaggle.json...")
        uploaded = files.upload()
        json_files = [name for name in uploaded if name.lower().endswith(".json")]
        if not json_files:
            raise FileNotFoundError("No JSON file was uploaded.")
        source = Path(json_files[0])
        shutil.copy2(source, target)
        os.chmod(target, 0o600)
        print("Kaggle credentials configured.")
    else:
        raise FileNotFoundError(
            "kaggle.json was not found. Place it in the current directory "
            "or at ~/.kaggle/kaggle.json."
        )

def run_command(command: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    print("[CMD]", " ".join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

def parse_kernel_dataset_sources(metadata_dir: Path) -> List[str]:
    candidates = list(metadata_dir.rglob("*metadata*.json")) + list(metadata_dir.rglob("kernel-metadata.json"))
    slugs: List[str] = []
    for metadata_file in candidates:
        try:
            data = json.loads(metadata_file.read_text(encoding="utf-8"))
        except Exception:
            continue
        for key in ["dataset_sources", "datasetSources"]:
            values = data.get(key, [])
            if isinstance(values, list):
                for item in values:
                    if isinstance(item, str) and "/" in item:
                        slugs.append(item)
                    elif isinstance(item, dict):
                        ref = item.get("ref") or item.get("source") or item.get("dataset")
                        if isinstance(ref, str) and "/" in ref:
                            slugs.append(ref)
        # Newer metadata may use "data_sources".
        for item in data.get("data_sources", []) if isinstance(data.get("data_sources", []), list) else []:
            if isinstance(item, dict):
                ref = item.get("ref") or item.get("source")
                source_type = str(item.get("sourceType", item.get("type", ""))).lower()
                if isinstance(ref, str) and "/" in ref and ("dataset" in source_type or source_type == ""):
                    slugs.append(ref)
    return sorted(set(slugs))

def discover_kernel_inputs(kernel_ref: str) -> List[str]:
    metadata_dir = WORK_DIR / "kaggle_kernel_metadata"
    if metadata_dir.exists():
        shutil.rmtree(metadata_dir)
    metadata_dir.mkdir(parents=True, exist_ok=True)

    commands = [
        ["kaggle", "kernels", "pull", kernel_ref, "-p", str(metadata_dir), "-m"],
        ["kaggle", "kernels", "pull", "-p", str(metadata_dir), "-m", kernel_ref],
    ]
    output = ""
    success = False
    for cmd in commands:
        try:
            result = run_command(cmd, check=True)
            output += result.stdout or ""
            success = True
            break
        except Exception as exc:
            output += f"\n{exc}"
    if not success:
        print("Could not pull Kaggle notebook metadata.")
        print(output[-2000:])
        return []

    slugs = parse_kernel_dataset_sources(metadata_dir)
    print("Discovered Kaggle dataset sources:", slugs)
    return slugs

def download_kaggle_dataset(slug: str, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / f".downloaded_{slug.replace('/', '__')}"
    if marker.exists() and any(destination.rglob("*")):
        print("Dataset already downloaded:", slug)
        return
    result = run_command([
        "kaggle", "datasets", "download",
        "-d", slug,
        "-p", str(destination),
        "--unzip",
    ])
    print(result.stdout[-3000:])
    marker.touch()

def download_data() -> List[str]:
    configure_kaggle_credentials()
    dataset_slugs: List[str] = []

    if KAGGLE_DATASET_SLUG.strip():
        dataset_slugs = [KAGGLE_DATASET_SLUG.strip()]
    else:
        dataset_slugs = discover_kernel_inputs(KAGGLE_KERNEL_REF)

    if not dataset_slugs:
        raise RuntimeError(
            "\nNo attached Kaggle dataset slug could be detected automatically.\n"
            "Open the cited Kaggle notebook's Input tab, copy the dataset slug in\n"
            "'owner/dataset-name' format, and set KAGGLE_DATASET_SLUG near the\n"
            "top of this script.\n"
        )

    for slug in dataset_slugs:
        download_kaggle_dataset(slug, DATA_DIR / slug.replace("/", "__"))
    return dataset_slugs

# ---------------------------------------------------------------------
# 5. DATASET DISCOVERY
# ---------------------------------------------------------------------
def normalize_token(text: str) -> str:
    return text.lower().replace(" ", "_").replace("-", "_")

def infer_label_from_path(path: Path) -> Optional[str]:
    components = [normalize_token(part) for part in path.parts]
    # Prioritize exact directory-name matches.
    for label in TARGET_CLASS_ORDER:
        aliases = [normalize_token(x) for x in CLASS_ALIASES[label]]
        for component in reversed(components[:-1]):
            if component == label or component in aliases:
                return label
    # Then allow contained tokens, with "malignant" checked before generic tumor.
    joined = "/".join(components)
    for label in ["malignant", "benign", "normal"]:
        for alias in CLASS_ALIASES[label]:
            alias_n = normalize_token(alias)
            if alias_n in joined:
                return label
    return None

def is_valid_image(path: Path) -> Tuple[bool, str]:
    """Validate an image with OpenCV before it enters tf.data.

    Some Kaggle folders contain files with an image extension but corrupted,
    truncated, or non-image content. tf.io.decode_image raises an
    InvalidArgumentError for such files and stops training. OpenCV validation
    lets us skip them safely and write a diagnostic CSV.
    """
    try:
        raw = np.fromfile(str(path), dtype=np.uint8)
        if raw.size == 0:
            return False, "empty_file"
        image = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if image is None:
            return False, "opencv_decode_failed"
        if image.ndim != 3 or image.shape[0] < 8 or image.shape[1] < 8:
            return False, f"invalid_shape_{getattr(image, 'shape', None)}"
        return True, "ok"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"

def collect_images(root: Path) -> pd.DataFrame:
    rows = []
    bad_rows = []
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            label = infer_label_from_path(path)
            if label is None:
                continue
            valid, reason = is_valid_image(path)
            if valid:
                rows.append({"filepath": str(path), "label": label})
            else:
                bad_rows.append({"filepath": str(path), "label": label, "reason": reason})

    if bad_rows:
        bad_df = pd.DataFrame(bad_rows)
        bad_df.to_csv(METRICS_DIR / "skipped_corrupt_images.csv", index=False)
        print(f"Skipped {len(bad_df)} corrupted/unreadable image files.")
        print(bad_df.head(10).to_string(index=False))
    else:
        pd.DataFrame(columns=["filepath", "label", "reason"]).to_csv(
            METRICS_DIR / "skipped_corrupt_images.csv", index=False
        )

    df = pd.DataFrame(rows).drop_duplicates("filepath") if rows else pd.DataFrame(columns=["filepath", "label"])
    if df.empty:
        dirs = sorted({p.name for p in root.rglob("*") if p.is_dir()})
        print("Available directories (first 100):", dirs[:100])
        raise RuntimeError(
            "No valid images could be mapped to normal/benign/malignant classes. "
            "Update CLASS_ALIASES according to the downloaded folder names."
        )

    present = sorted(df["label"].unique().tolist())
    print("\nDetected classes:", present)
    print(df["label"].value_counts())
    if len(present) < 2:
        raise RuntimeError("At least two classes are required.")
    return df.reset_index(drop=True)

def create_splits(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    min_count = int(df["label"].value_counts().min())
    stratify_all = df["label"] if min_count >= 3 else None

    train_val, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=stratify_all,
    )
    min_remaining = int(train_val["label"].value_counts().min())
    stratify_remaining = train_val["label"] if min_remaining >= 3 else None

    train_df, val_df = train_test_split(
        train_val,
        test_size=VAL_SIZE_FROM_REMAINING,
        random_state=SEED,
        stratify=stratify_remaining,
    )

    for name, split in [("train", train_df), ("validation", val_df), ("test", test_df)]:
        split.to_csv(METRICS_DIR / f"{name}_split.csv", index=False)
        print(f"\n{name.upper()} ({len(split)} images)")
        print(split["label"].value_counts())

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

# ---------------------------------------------------------------------
# 6. PREPROCESSING
# ---------------------------------------------------------------------
def read_rgb_image(path: str) -> np.ndarray:
    image_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Unable to read image: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

def paper_preprocess_numpy(image_rgb: np.ndarray) -> np.ndarray:
    image_rgb = np.asarray(image_rgb, dtype=np.uint8)
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    if USE_BILATERAL_FILTER:
        gray = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

    if USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)

    if USE_PSEUDOCOLOR:
        color_bgr = cv2.applyColorMap(gray, cv2.COLORMAP_JET)
        processed = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2RGB)
    else:
        processed = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

    processed = cv2.resize(processed, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return processed.astype(np.float32)

def load_preprocess_from_path_numpy(path_value: Any) -> np.ndarray:
    """Decode by OpenCV from a path and apply the paper preprocessing.

    Using cv2.imdecode rather than tf.io.decode_image avoids TensorFlow PNG
    decoder crashes caused by malformed PNG metadata in a few Kaggle files.
    Files have already been validated in collect_images; this function retains
    a defensive zero-image fallback so one unexpected read error cannot stop
    an entire training run.
    """
    try:
        if isinstance(path_value, np.ndarray):
            path_value = path_value.item()
        if isinstance(path_value, (bytes, np.bytes_)):
            path_str = path_value.decode("utf-8")
        else:
            path_str = str(path_value)
        raw = np.fromfile(path_str, dtype=np.uint8)
        image_bgr = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise ValueError("OpenCV could not decode the image")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        processed = paper_preprocess_numpy(image_rgb)
        return efficientnetv2_preprocess(processed.copy()).astype(np.float32)
    except Exception as exc:
        print(f"[WARNING] Runtime image read failed: {path_value!r} | {exc}")
        fallback = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        return efficientnetv2_preprocess(fallback).astype(np.float32)

def load_and_preprocess(path: tf.Tensor, label: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    image = tf.numpy_function(load_preprocess_from_path_numpy, [path], tf.float32)
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    label = tf.cast(label, tf.int32)
    return image, label

def make_augmentation() -> keras.Sequential:
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical", seed=SEED),
            layers.RandomRotation(20.0 / 360.0, fill_mode="reflect", seed=SEED),
            layers.RandomTranslation(0.06, 0.06, fill_mode="reflect", seed=SEED),
            layers.RandomZoom(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10), seed=SEED),
            layers.RandomContrast(0.10, seed=SEED),
        ],
        name="training_augmentation",
    )

def build_dataset(
    df: pd.DataFrame,
    class_to_index: Dict[str, int],
    training: bool = False,
) -> tf.data.Dataset:
    paths = df["filepath"].astype(str).values
    labels = df["label"].map(class_to_index).astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=max(len(df), 1), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    # Additional protection against rare I/O failures in remote notebook runtimes.
    ds = ds.apply(tf.data.experimental.ignore_errors(log_warning=True))
    if training:
        augmenter = make_augmentation()
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def save_preprocessing_samples(df: pd.DataFrame, n: int = 9) -> None:
    sample_df = df.sample(min(n, len(df)), random_state=SEED)
    rows = len(sample_df)
    fig, axes = plt.subplots(rows, 2, figsize=(8, max(3 * rows, 6)))
    if rows == 1:
        axes = np.array([axes])
    for i, (_, row) in enumerate(sample_df.iterrows()):
        original = read_rgb_image(row["filepath"])
        processed = paper_preprocess_numpy(original).astype(np.uint8)
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f"Before: {row['label']}", fontsize=10, fontweight="bold")
        axes[i, 1].imshow(processed)
        axes[i, 1].set_title(f"After: {row['label']}", fontsize=10, fontweight="bold")
        axes[i, 0].axis("off")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.savefig(SAMPLES_DIR / "before_after_preprocessing_samples.png", dpi=300, bbox_inches="tight")
    plt.close()

# ---------------------------------------------------------------------
# 7. MODEL
# ---------------------------------------------------------------------
def build_efficientnetv2s(num_classes: int) -> Tuple[keras.Model, keras.Model]:
    """Build ImageNet-pretrained EfficientNetV2-S for liver tumor classification.

    EfficientNetV2-S includes its own input rescaling layer by default. The
    preceding custom CT preprocessing returns values in [0, 255], which is the
    expected input range when include_preprocessing=True.
    """
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="ct_image")

    backbone = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling=None,
        include_preprocessing=True,
    )
    backbone.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.BatchNormalization(name="head_batch_norm")(x)
    x = layers.Dropout(0.40, name="head_dropout")(x)

    # Keep output in float32 when mixed precision is active.
    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="EfficientNetV2S_Liver_Tumor",
    )
    return model, backbone


def compile_model(model: keras.Model, learning_rate: float) -> None:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

def unfreeze_backbone(backbone: keras.Model, last_n_layers: int = 40) -> None:
    backbone.trainable = True
    for layer in backbone.layers[:-last_n_layers]:
        layer.trainable = False
    # Keep BatchNorm frozen for stable fine-tuning on a small medical dataset.
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

# ---------------------------------------------------------------------
# 8. METRICS
# ---------------------------------------------------------------------
def safe_multiclass_auc(y_true: np.ndarray, y_prob: np.ndarray, num_classes: int) -> float:
    try:
        if num_classes == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1]))
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")

def specificity_per_class(cm: np.ndarray) -> np.ndarray:
    total = cm.sum()
    values = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        values.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return np.asarray(values, dtype=float)

def per_class_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray, class_names: List[str]) -> pd.DataFrame:
    num_classes = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(num_classes), zero_division=0
    )
    specificity = specificity_per_class(cm)
    class_accuracy = []
    aucs = []
    y_bin = label_binarize(y_true, classes=np.arange(num_classes))
    if num_classes == 2 and y_bin.ndim == 2 and y_bin.shape[1] == 1:
        y_bin = np.concatenate([1 - y_bin, y_bin], axis=1)

    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        class_accuracy.append((tp + tn) / cm.sum() if cm.sum() else np.nan)
        try:
            aucs.append(roc_auc_score(y_bin[:, i], y_prob[:, i]))
        except Exception:
            aucs.append(np.nan)

    return pd.DataFrame({
        "class": class_names,
        "accuracy_ovr": class_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "auc_ovr": aucs,
        "support": support,
    })

def compute_split_metrics(
    model: keras.Model,
    ds: tf.data.Dataset,
    split_name: str,
    class_names: List[str],
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    eval_result = model.evaluate(ds, verbose=0, return_dict=True)
    y_true_parts, y_prob_parts = [], []
    for x_batch, y_batch in ds:
        probs = model.predict_on_batch(x_batch)
        y_true_parts.append(y_batch.numpy())
        y_prob_parts.append(np.asarray(probs))
    y_true = np.concatenate(y_true_parts)
    y_prob = np.concatenate(y_prob_parts)
    y_pred = np.argmax(y_prob, axis=1)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    specificity_macro = float(np.nanmean(specificity_per_class(cm)))
    auc_macro = safe_multiclass_auc(y_true, y_prob, len(class_names))

    metrics = {
        "split": split_name,
        "loss": float(eval_result["loss"]),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "sensitivity_macro": float(recall_macro),
        "specificity_macro": specificity_macro,
        "f1_macro": float(f1_macro),
        "auc_macro_ovr": auc_macro,
        "n_samples": int(len(y_true)),
    }

    class_df = per_class_metrics(y_true, y_pred, y_prob, class_names)
    class_df.insert(0, "split", split_name)

    pd.DataFrame([metrics]).to_csv(METRICS_DIR / f"{split_name}_overall_metrics.csv", index=False)
    class_df.to_csv(METRICS_DIR / f"{split_name}_per_class_metrics.csv", index=False)

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(METRICS_DIR / f"{split_name}_classification_report.csv")

    pred_df = pd.DataFrame({
        "true_index": y_true,
        "pred_index": y_pred,
        "true_class": [class_names[i] for i in y_true],
        "pred_class": [class_names[i] for i in y_pred],
        "predicted_confidence": np.max(y_prob, axis=1),
        "correct": (y_true == y_pred).astype(int),
    })
    for i, name in enumerate(class_names):
        pred_df[f"prob_{name}"] = y_prob[:, i]
    pred_df.to_csv(METRICS_DIR / f"{split_name}_predictions.csv", index=False)

    return metrics, y_true, y_pred, y_prob, class_df

# ---------------------------------------------------------------------
# 9. PER-EPOCH TEST CALLBACK
# ---------------------------------------------------------------------
class TestMetricsCallback(keras.callbacks.Callback):
    def __init__(self, test_ds: tf.data.Dataset, class_names: List[str]):
        super().__init__()
        self.test_ds = test_ds
        self.class_names = class_names
        self.records: List[Dict[str, float]] = []

    def on_epoch_end(self, epoch: int, logs: Optional[Dict[str, Any]] = None) -> None:
        logs = logs or {}
        result = self.model.evaluate(self.test_ds, verbose=0, return_dict=True)
        y_true_parts, y_prob_parts = [], []
        for xb, yb in self.test_ds:
            pb = self.model.predict_on_batch(xb)
            y_true_parts.append(yb.numpy())
            y_prob_parts.append(np.asarray(pb))
        y_true = np.concatenate(y_true_parts)
        y_prob = np.concatenate(y_prob_parts)
        y_pred = np.argmax(y_prob, axis=1)
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(self.class_names)))
        spec = float(np.nanmean(specificity_per_class(cm)))
        auc_value = safe_multiclass_auc(y_true, y_prob, len(self.class_names))
        record = {
            "epoch": epoch + 1,
            "test_loss": float(result["loss"]),
            "test_accuracy": float(accuracy_score(y_true, y_pred)),
            "test_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_sensitivity": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_specificity": spec,
            "test_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "test_auc": auc_value,
        }
        self.records.append(record)
        print(
            f" — test_acc: {record['test_accuracy']:.4f}"
            f" — test_prec: {record['test_precision']:.4f}"
            f" — test_rec/sens: {record['test_recall']:.4f}"
            f" — test_spec: {record['test_specificity']:.4f}"
            f" — test_auc: {record['test_auc']:.4f}"
        )

# ---------------------------------------------------------------------
# 10. PLOTS
# ---------------------------------------------------------------------
def save_confusion_matrix(cm: np.ndarray, class_names: List[str], split_name: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="turbo")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(class_names)), class_names, rotation=35, ha="right")
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("Predicted label", fontweight="bold")
    ax.set_ylabel("True label", fontweight="bold")
    ax.set_title(f"{split_name.title()} Confusion Matrix", fontweight="bold")
    threshold = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > threshold else "black", fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_roc_curves(y_true: np.ndarray, y_prob: np.ndarray, class_names: List[str], split_name: str) -> None:
    num_classes = len(class_names)
    y_bin = label_binarize(y_true, classes=np.arange(num_classes))
    if num_classes == 2 and y_bin.shape[1] == 1:
        y_bin = np.concatenate([1 - y_bin, y_bin], axis=1)

    fig, ax = plt.subplots(figsize=(8, 7))
    colors = plt.cm.hsv(np.linspace(0, 0.85, num_classes))
    for i, (name, color) in enumerate(zip(class_names, colors)):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, linewidth=2.5, color=color, label=f"{name} (AUC={roc_auc:.3f})")
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, color="black")
    ax.set_xlabel("False Positive Rate", fontweight="bold")
    ax.set_ylabel("True Positive Rate", fontweight="bold")
    ax.set_title(f"{split_name.title()} ROC–AUC Curves", fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_roc_auc.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_metric_boxplot(class_df: pd.DataFrame, split_name: str) -> None:
    columns = ["accuracy_ovr", "precision", "recall_sensitivity", "specificity", "f1_score", "auc_ovr"]
    labels = ["Accuracy", "Precision", "Recall/\nSensitivity", "Specificity", "F1", "AUC"]
    data = [class_df[col].dropna().values for col in columns]
    fig, ax = plt.subplots(figsize=(10, 6))
    box = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(box["boxes"])))
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Metric value", fontweight="bold")
    ax.set_title(f"{split_name.title()} Per-Class Metric Box Plot", fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_metrics_boxplot.png", dpi=300, bbox_inches="tight")
    plt.close()

def save_confidence_boxplot(y_true: np.ndarray, y_prob: np.ndarray, class_names: List[str], split_name: str) -> None:
    confidences = np.max(y_prob, axis=1)
    data = [confidences[y_true == i] for i in range(len(class_names))]
    fig, ax = plt.subplots(figsize=(8, 6))
    box = ax.boxplot(data, labels=class_names, patch_artist=True, showmeans=True)
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(box["boxes"])))
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Predicted confidence", fontweight="bold")
    ax.set_title(f"{split_name.title()} Confidence Distribution", fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split_name}_confidence_boxplot.png", dpi=300, bbox_inches="tight")
    plt.close()

def merge_histories(history1: keras.callbacks.History, history2: keras.callbacks.History) -> Dict[str, List[float]]:
    merged: Dict[str, List[float]] = {}
    keys = set(history1.history.keys()) | set(history2.history.keys())
    for key in keys:
        merged[key] = list(history1.history.get(key, [])) + list(history2.history.get(key, []))
    return merged

def save_training_curves(history: Dict[str, List[float]], test_records: List[Dict[str, float]]) -> None:
    epochs = np.arange(1, len(history.get("accuracy", [])) + 1)
    test_df = pd.DataFrame(test_records)
    history_df = pd.DataFrame(history)
    history_df.insert(0, "epoch", epochs)
    history_df.to_csv(METRICS_DIR / "training_history.csv", index=False)
    test_df.to_csv(METRICS_DIR / "per_epoch_test_metrics.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, history["accuracy"], linewidth=2.5, color=BRIGHT_COLORS["train"], label="Training")
    ax.plot(epochs, history["val_accuracy"], linewidth=2.5, color=BRIGHT_COLORS["validation"], label="Validation")
    if not test_df.empty:
        ax.plot(test_df["epoch"], test_df["test_accuracy"], linewidth=2.5,
                color=BRIGHT_COLORS["test"], label="Test")
    ax.axvline(HEAD_EPOCHS, linestyle="--", color="black", alpha=0.7, label="Fine-tuning starts")
    ax.set_xlabel("Epoch", fontweight="bold")
    ax.set_ylabel("Accuracy", fontweight="bold")
    ax.set_title("EfficientNetV2-S Accuracy Curves", fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "accuracy_curves_train_validation_test.png", dpi=300, bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, history["loss"], linewidth=2.5, color=BRIGHT_COLORS["loss_train"], label="Training")
    ax.plot(epochs, history["val_loss"], linewidth=2.5, color=BRIGHT_COLORS["loss_validation"], label="Validation")
    if not test_df.empty:
        ax.plot(test_df["epoch"], test_df["test_loss"], linewidth=2.5,
                color=BRIGHT_COLORS["loss_test"], label="Test")
    ax.axvline(HEAD_EPOCHS, linestyle="--", color="black", alpha=0.7, label="Fine-tuning starts")
    ax.set_xlabel("Epoch", fontweight="bold")
    ax.set_ylabel("Loss", fontweight="bold")
    ax.set_title("EfficientNetV2-S Loss Curves", fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "loss_curves_train_validation_test.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Test metric curves.
    if not test_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        metric_cols = [
            ("test_accuracy", "Accuracy"),
            ("test_precision", "Precision"),
            ("test_recall", "Recall/Sensitivity"),
            ("test_specificity", "Specificity"),
            ("test_auc", "AUC"),
        ]
        colors = plt.cm.hsv(np.linspace(0, 0.85, len(metric_cols)))
        for (col, label), color in zip(metric_cols, colors):
            ax.plot(test_df["epoch"], test_df[col], linewidth=2.2, label=label, color=color)
        ax.set_xlabel("Epoch", fontweight="bold")
        ax.set_ylabel("Metric value", fontweight="bold")
        ax.set_ylim(0, 1.05)
        ax.set_title("Per-Epoch Test Metrics", fontweight="bold")
        ax.legend(ncol=2)
        ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "per_epoch_test_metric_curves.png", dpi=300, bbox_inches="tight")
        plt.close()

# ---------------------------------------------------------------------
# 11. MODEL COMPLEXITY
# ---------------------------------------------------------------------
def count_parameters(model: keras.Model) -> Dict[str, int]:
    total = int(model.count_params())
    trainable = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    non_trainable = int(sum(np.prod(v.shape) for v in model.non_trainable_weights))
    return {
        "total_parameters": total,
        "trainable_parameters": trainable,
        "non_trainable_parameters": non_trainable,
    }

def estimate_flops(model: keras.Model) -> Tuple[Optional[int], Optional[float]]:
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        concrete = tf.function(lambda x: model(x, training=False)).get_concrete_function(
            tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32)
        )
        frozen = convert_variables_to_constants_v2(concrete)
        graph_def = frozen.graph.as_graph_def()
        with tf.Graph().as_default() as graph:
            tf.compat.v1.graph_util.import_graph_def(graph_def, name="")
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            profile = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd="op", options=opts)
            flops = int(profile.total_float_ops) if profile is not None else None
        return flops, (flops / 1e9 if flops is not None else None)
    except Exception as exc:
        print("FLOPs estimation failed:", exc)
        return None, None

# ---------------------------------------------------------------------
# 12. XAI HELPERS
# ---------------------------------------------------------------------
def last_conv_layer_name(model: keras.Model) -> str:
    """Return a classifier-connected 4-D EfficientNetV2-S feature layer."""
    preferred_layers = [
        "top_activation",
        "top_conv",
        "block6s_add",
        "block6s_project_conv",
    ]

    for candidate in preferred_layers:
        try:
            layer = model.get_layer(candidate)
            if len(layer.output.shape) == 4:
                return candidate
        except Exception:
            continue

    # Generic fallback: select the last 4-D layer directly connected in model.
    for layer in reversed(model.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except Exception:
            continue

    raise ValueError("No suitable 4-D EfficientNetV2-S feature layer was found.")


def normalize_heatmap(heatmap: np.ndarray) -> np.ndarray:
    """Return a safe, finite, contiguous float32 2-D heatmap in [0, 1].

    Mixed precision can produce float16 heatmaps, while OpenCV resize does not
    reliably support float16. This helper also handles singleton dimensions,
    NaN/Inf values, and degenerate all-zero maps.
    """
    heatmap = np.asarray(heatmap)
    heatmap = np.squeeze(heatmap)

    if heatmap.ndim == 3:
        # Collapse an unexpected channel dimension safely.
        heatmap = np.mean(np.abs(heatmap.astype(np.float32)), axis=-1)
    if heatmap.ndim != 2:
        raise ValueError(f"Expected a 2-D heatmap, received shape {heatmap.shape}")

    heatmap = np.ascontiguousarray(heatmap, dtype=np.float32)
    heatmap = np.nan_to_num(heatmap, nan=0.0, posinf=0.0, neginf=0.0)
    heatmap = np.maximum(heatmap, 0.0)
    maximum = float(np.max(heatmap)) if heatmap.size else 0.0
    if maximum <= 1e-12:
        return np.zeros_like(heatmap, dtype=np.float32)
    return np.ascontiguousarray(heatmap / maximum, dtype=np.float32)

def gradcam(model: keras.Model, image_batch: tf.Tensor, class_index: int, layer_name: str) -> np.ndarray:
    """Generate Grad-CAM without breaking the gradient path under mixed precision.

    Important: gradients must be requested with respect to the original tensor
    returned by the model. Casting that tensor inside GradientTape and then asking
    for gradients with respect to the cast tensor disconnects the graph and returns
    None. We therefore differentiate first and cast only afterwards.
    """
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output],
        name="gradcam_model",
    )

    image_batch = tf.cast(image_batch, model.inputs[0].dtype)
    with tf.GradientTape() as tape:
        raw_conv_output, raw_predictions = grad_model(image_batch, training=False)
        # Watch explicitly for compatibility with loaded Keras 3 models.
        tape.watch(raw_conv_output)
        predictions_f32 = tf.cast(raw_predictions, tf.float32)
        target_score = predictions_f32[:, int(class_index)]

    grads = tape.gradient(target_score, raw_conv_output)
    if grads is None:
        raise RuntimeError(
            f"Grad-CAM gradients are None for layer '{layer_name}'. "
            "Use a convolutional layer connected to the classifier output."
        )

    conv_output = tf.cast(raw_conv_output, tf.float32)
    grads = tf.cast(grads, tf.float32)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv_output, axis=-1)[0]
    return normalize_heatmap(cam.numpy())


def gradcam_plus_plus(model: keras.Model, image_batch: tf.Tensor, class_index: int, layer_name: str) -> np.ndarray:
    """Generate a numerically stable Grad-CAM++ map under mixed precision."""
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output],
        name="gradcam_plus_plus_model",
    )

    image_batch = tf.cast(image_batch, model.inputs[0].dtype)
    with tf.GradientTape() as tape:
        raw_conv_output, raw_predictions = grad_model(image_batch, training=False)
        tape.watch(raw_conv_output)
        predictions_f32 = tf.cast(raw_predictions, tf.float32)
        score = predictions_f32[:, int(class_index)]

    raw_grads = tape.gradient(score, raw_conv_output)
    if raw_grads is None:
        raise RuntimeError(
            f"Grad-CAM++ gradients are None for layer '{layer_name}'. "
            "Use a convolutional layer connected to the classifier output."
        )

    conv = tf.cast(raw_conv_output[0], tf.float32)
    grads = tf.cast(raw_grads[0], tf.float32)

    grads2 = tf.square(grads)
    grads3 = grads2 * grads
    spatial_sum = tf.reduce_sum(conv, axis=(0, 1), keepdims=True)
    denominator = 2.0 * grads2 + spatial_sum * grads3
    denominator = tf.where(
        tf.abs(denominator) > 1e-10,
        denominator,
        tf.ones_like(denominator),
    )
    alphas = grads2 / denominator
    alphas = tf.where(tf.math.is_finite(alphas), alphas, tf.zeros_like(alphas))

    positive_grads = tf.nn.relu(grads)
    weights = tf.reduce_sum(alphas * positive_grads, axis=(0, 1))
    cam = tf.reduce_sum(conv * weights[None, None, :], axis=-1)
    return normalize_heatmap(cam.numpy())

def saliency_map(model: keras.Model, image_batch: tf.Tensor, class_index: int) -> np.ndarray:
    image_var = tf.Variable(tf.cast(image_batch, tf.float32))
    with tf.GradientTape() as tape:
        predictions = tf.cast(model(image_var, training=False), tf.float32)
        score = predictions[:, class_index]
    gradients = tape.gradient(score, image_var)
    if gradients is None:
        raise RuntimeError("Saliency gradients are None.")
    saliency = tf.reduce_max(tf.abs(tf.cast(gradients[0], tf.float32)), axis=-1).numpy()
    return normalize_heatmap(saliency)

def heatmap_overlay(display_rgb: np.ndarray, heatmap: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Resize and overlay a heatmap without OpenCV float16 failures."""
    display_rgb = np.asarray(display_rgb)
    if display_rgb.ndim != 3 or display_rgb.shape[-1] != 3:
        raise ValueError(f"Expected RGB image with shape (H,W,3), got {display_rgb.shape}")
    display_rgb = np.ascontiguousarray(np.clip(display_rgb, 0, 255), dtype=np.uint8)

    safe_heatmap = normalize_heatmap(heatmap)
    target_size = (int(display_rgb.shape[1]), int(display_rgb.shape[0]))
    resized = cv2.resize(
        np.ascontiguousarray(safe_heatmap, dtype=np.float32),
        target_size,
        interpolation=cv2.INTER_CUBIC,
    )
    resized = np.nan_to_num(resized, nan=0.0, posinf=1.0, neginf=0.0)
    resized = np.clip(resized, 0.0, 1.0)
    heatmap_uint8 = np.ascontiguousarray(np.rint(resized * 255.0), dtype=np.uint8)

    colored_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(display_rgb, 1.0 - float(alpha), colored_rgb, float(alpha), 0.0)
    return np.ascontiguousarray(overlay, dtype=np.uint8)

def model_ready_from_rgb(rgb_uint8: np.ndarray) -> np.ndarray:
    processed = paper_preprocess_numpy(rgb_uint8)
    return efficientnetv2_preprocess(processed.copy()).astype(np.float32)

def lime_explanation(
    model: keras.Model,
    display_rgb: np.ndarray,
    class_index: int,
) -> np.ndarray:
    explainer = lime_image.LimeImageExplainer(random_state=SEED)

    def predict_fn(images: np.ndarray) -> np.ndarray:
        batch = np.stack([model_ready_from_rgb(np.asarray(img, dtype=np.uint8)) for img in images])
        return model.predict(batch, batch_size=XAI_BATCH_SIZE, verbose=0)

    explanation = explainer.explain_instance(
        display_rgb.astype(np.double),
        classifier_fn=predict_fn,
        top_labels=max(3, class_index + 1),
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES,
        random_seed=SEED,
    )
    temp, mask = explanation.get_image_and_mask(
        class_index,
        positive_only=True,
        num_features=10,
        hide_rest=False,
    )
    temp = np.clip(temp, 0, 255).astype(np.uint8)
    boundary = mark_boundaries(temp / 255.0, mask)
    return np.uint8(np.clip(boundary, 0, 1) * 255)

def prepare_shap_explainer(model: keras.Model, background_batch: np.ndarray):
    try:
        return shap.GradientExplainer(model, background_batch)
    except Exception as exc:
        print("GradientExplainer failed; using DeepExplainer:", exc)
        return shap.DeepExplainer(model, background_batch)

def extract_shap_map(shap_values: Any, class_index: int) -> np.ndarray:
    # SHAP versions return either list[num_classes] of (B,H,W,C),
    # or one array (B,H,W,C,num_classes).
    if isinstance(shap_values, list):
        arr = np.asarray(shap_values[min(class_index, len(shap_values) - 1)])
        sample = arr[0]
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 5:
            sample = arr[0, ..., min(class_index, arr.shape[-1] - 1)]
        elif arr.ndim == 4:
            sample = arr[0]
        else:
            sample = np.squeeze(arr)
    if sample.ndim == 3:
        sample = np.mean(np.abs(sample), axis=-1)
    return normalize_heatmap(np.asarray(sample, dtype=np.float32))

def save_xai_for_first_test_images(
    model: keras.Model,
    test_df: pd.DataFrame,
    class_to_index: Dict[str, int],
    class_names: List[str],
    n_images: int = N_XAI,
) -> None:
    if test_df.empty:
        return

    technique_dirs = {}
    for technique in ["gradcam", "gradcam_plus_plus", "lime", "shap", "saliency", "combined"]:
        technique_dirs[technique] = XAI_DIR / technique
        technique_dirs[technique].mkdir(parents=True, exist_ok=True)

    layer_name = last_conv_layer_name(model)
    print("XAI convolutional layer:", layer_name)

    # SHAP background from the first available test images.
    bg_images = []
    for path in test_df["filepath"].head(SHAP_BACKGROUND_SIZE):
        rgb = read_rgb_image(path)
        bg_images.append(model_ready_from_rgb(rgb))
    background = np.stack(bg_images).astype(np.float32)
    shap_explainer = prepare_shap_explainer(model, background)

    xai_rows = []
    for number, (_, row) in enumerate(test_df.head(n_images).iterrows(), start=1):
        print(f"Generating XAI {number}/{min(n_images, len(test_df))}: {row['filepath']}")
        original = read_rgb_image(row["filepath"])
        display_rgb = cv2.resize(original, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        model_input = model_ready_from_rgb(original)
        batch = tf.convert_to_tensor(model_input[None, ...], dtype=tf.float32)
        probs = model.predict(batch, verbose=0)[0]
        pred_index = int(np.argmax(probs))
        true_index = class_to_index[row["label"]]

        # Grad-CAM.
        gc_heat = gradcam(model, batch, pred_index, layer_name)
        gc_img = heatmap_overlay(display_rgb, gc_heat)

        # Grad-CAM++.
        try:
            gcpp_heat = gradcam_plus_plus(model, batch, pred_index, layer_name)
            gcpp_img = heatmap_overlay(display_rgb, gcpp_heat)
        except Exception as exc:
            print("Grad-CAM++ fallback to Grad-CAM:", exc)
            gcpp_heat = gc_heat
            gcpp_img = gc_img

        # Saliency.
        sal_heat = saliency_map(model, batch, pred_index)
        sal_img = heatmap_overlay(display_rgb, sal_heat)

        # LIME.
        try:
            lime_img = lime_explanation(model, display_rgb, pred_index)
        except Exception as exc:
            print("LIME failed:", exc)
            lime_img = display_rgb.copy()

        # SHAP.
        try:
            shap_values = shap_explainer.shap_values(model_input[None, ...])
            shap_heat = extract_shap_map(shap_values, pred_index)
            shap_img = heatmap_overlay(display_rgb, shap_heat)
        except Exception as exc:
            print("SHAP failed:", exc)
            shap_img = display_rgb.copy()

        stem = f"Patient_{number:02d}_true_{class_names[true_index]}_pred_{class_names[pred_index]}"
        for technique, image in [
            ("gradcam", gc_img),
            ("gradcam_plus_plus", gcpp_img),
            ("lime", lime_img),
            ("shap", shap_img),
            ("saliency", sal_img),
        ]:
            cv2.imwrite(
                str(technique_dirs[technique] / f"{stem}.png"),
                cv2.cvtColor(image.astype(np.uint8), cv2.COLOR_RGB2BGR),
            )

        # Combined six-panel image.
        fig, axes = plt.subplots(2, 3, figsize=(13, 9))
        images = [display_rgb, gc_img, gcpp_img, lime_img, shap_img, sal_img]
        titles = [
            "Original",
            "Grad-CAM",
            "Grad-CAM++",
            "LIME",
            "SHAP",
            "Saliency Map",
        ]
        for ax, image, title in zip(axes.ravel(), images, titles):
            ax.imshow(image)
            ax.set_title(title, fontweight="bold")
            ax.axis("off")
        fig.suptitle(
            f"Patient {number:02d} | True: {class_names[true_index]} | "
            f"Predicted: {class_names[pred_index]} ({probs[pred_index]:.4f})",
            fontsize=13,
            fontweight="bold",
        )
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        combined_path = technique_dirs["combined"] / f"{stem}.png"
        plt.savefig(combined_path, dpi=250, bbox_inches="tight")
        plt.close()

        record = {
            "patient_number": number,
            "filepath": row["filepath"],
            "true_class": class_names[true_index],
            "predicted_class": class_names[pred_index],
            "confidence": float(probs[pred_index]),
            "correct": int(true_index == pred_index),
        }
        for idx, class_name in enumerate(class_names):
            record[f"prob_{class_name}"] = float(probs[idx])
        xai_rows.append(record)

        # Avoid graph accumulation during 20 repeated explanation runs.
        tf.keras.backend.clear_session if False else None

    pd.DataFrame(xai_rows).to_csv(XAI_DIR / "first_20_test_xai_predictions.csv", index=False)

# ---------------------------------------------------------------------
# 13. ZIP AND DOWNLOAD
# ---------------------------------------------------------------------
def zip_results() -> Path:
    zip_path = WORK_DIR / "EfficientNetV2S_Liver_Tumor_Complete_Results.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in RESULTS_DIR.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=file_path.relative_to(RESULTS_DIR.parent))
    return zip_path

def auto_download(path: Path) -> None:
    if in_colab():
        from google.colab import files
        print("Starting automatic download:", path)
        files.download(str(path))
    else:
        print("Results ZIP:", path.resolve())

# ---------------------------------------------------------------------
# 14. MAIN
# ---------------------------------------------------------------------
def main() -> None:
    start_total = time.perf_counter()

    # A. Download.
    slugs = download_data()
    (RESULTS_DIR / "downloaded_kaggle_sources.txt").write_text("\n".join(slugs), encoding="utf-8")

    # B. Discover and split.
    all_df = collect_images(DATA_DIR)
    class_names = [name for name in TARGET_CLASS_ORDER if name in all_df["label"].unique()]
    # Include unexpected mapped labels safely.
    class_names += sorted(set(all_df["label"].unique()) - set(class_names))
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    (RESULTS_DIR / "class_mapping.json").write_text(
        json.dumps(class_to_index, indent=2), encoding="utf-8"
    )

    train_df, val_df, test_df = create_splits(all_df)
    save_preprocessing_samples(train_df, n=9)

    train_ds = build_dataset(train_df, class_to_index, training=True)
    train_eval_ds = build_dataset(train_df, class_to_index, training=False)
    val_ds = build_dataset(val_df, class_to_index, training=False)
    test_ds = build_dataset(test_df, class_to_index, training=False)

    # C. Build and profile before training.
    model, backbone = build_efficientnetv2s(len(class_names))
    compile_model(model, INITIAL_LR)
    print(f"Model: {model.name}")
    print(f"Total parameters: {model.count_params():,}")
    print(f"Trainable parameters (stage 1): {sum(int(np.prod(v.shape)) for v in model.trainable_weights):,}")
    print(f"Non-trainable parameters (stage 1): {sum(int(np.prod(v.shape)) for v in model.non_trainable_weights):,}")
    with open(MODEL_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
        model.summary(print_fn=lambda line: f.write(line + "\n"))

    complexity = count_parameters(model)
    flops, gflops = estimate_flops(model)
    complexity["flops_per_forward_pass"] = flops
    complexity["gflops_per_forward_pass"] = gflops

    # D. Train frozen head.
    best_model_path = MODEL_DIR / "best_efficientnetv2s_liver.keras"
    callbacks_common = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=4,
            min_lr=1e-7,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(METRICS_DIR / "training_log.csv"), append=False),
    ]

    test_callback_stage1 = TestMetricsCallback(test_ds, class_names)
    training_start = time.perf_counter()
    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks_common + [test_callback_stage1],
        verbose=1,
    )

    # E. Fine-tune.
    unfreeze_backbone(backbone, last_n_layers=60)
    compile_model(model, FINE_TUNE_LR)

    callbacks_finetune = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=4,
            min_lr=1e-8,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(METRICS_DIR / "fine_tuning_log.csv"), append=False),
    ]
    test_callback_stage2 = TestMetricsCallback(test_ds, class_names)
    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=HEAD_EPOCHS,
        epochs=TOTAL_EPOCHS,
        callbacks=callbacks_finetune + [test_callback_stage2],
        verbose=1,
    )
    training_time_seconds = time.perf_counter() - training_start

    # Load best model.
    if best_model_path.exists():
        model = keras.models.load_model(best_model_path)

    merged_history = merge_histories(history1, history2)
    test_records = test_callback_stage1.records + test_callback_stage2.records
    save_training_curves(merged_history, test_records)

    # F. Final split evaluations.
    all_metrics = []
    eval_payload = {}
    for split_name, ds in [
        ("training", train_eval_ds),
        ("validation", val_ds),
        ("test", test_ds),
    ]:
        metrics, y_true, y_pred, y_prob, class_df = compute_split_metrics(
            model, ds, split_name, class_names
        )
        all_metrics.append(metrics)
        eval_payload[split_name] = (y_true, y_pred, y_prob, class_df)
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
        save_confusion_matrix(cm, class_names, split_name)
        save_roc_curves(y_true, y_prob, class_names, split_name)
        save_metric_boxplot(class_df, split_name)
        save_confidence_boxplot(y_true, y_prob, class_names, split_name)

    overall_df = pd.DataFrame(all_metrics)
    overall_df.to_csv(METRICS_DIR / "all_split_overall_metrics.csv", index=False)
    print("\nFINAL METRICS\n", overall_df.to_string(index=False))

    # G. Complexity and timing.
    complexity.update({
        "training_time_seconds": float(training_time_seconds),
        "training_time_minutes": float(training_time_seconds / 60.0),
        "total_pipeline_time_seconds_before_xai": float(time.perf_counter() - start_total),
        "input_shape": [1, IMG_SIZE, IMG_SIZE, 3],
        "batch_size": BATCH_SIZE,
        "head_epochs_requested": HEAD_EPOCHS,
        "fine_tune_epochs_requested": FINE_TUNE_EPOCHS,
    })
    with open(METRICS_DIR / "model_complexity_and_time.json", "w", encoding="utf-8") as f:
        json.dump(complexity, f, indent=2)
    pd.DataFrame([complexity]).to_csv(METRICS_DIR / "model_complexity_and_time.csv", index=False)

    # H. Save final model.
    model.save(MODEL_DIR / "final_efficientnetv2s_liver.keras")

    # I. XAI first 20 test images.
    save_xai_for_first_test_images(
        model=model,
        test_df=test_df,
        class_to_index=class_to_index,
        class_names=class_names,
        n_images=N_XAI,
    )

    # J. Final metadata and ZIP.
    final_metadata = {
        "base_model": "EfficientNetV2-S",
        "class_names": class_names,
        "class_mapping": class_to_index,
        "kaggle_sources": slugs,
        "dataset_images_total": int(len(all_df)),
        "training_images": int(len(train_df)),
        "validation_images": int(len(val_df)),
        "test_images": int(len(test_df)),
        "preprocessing": {
            "bilateral_filter": USE_BILATERAL_FILTER,
            "clahe": USE_CLAHE,
            "pseudocolor": USE_PSEUDOCOLOR,
            "resize": [IMG_SIZE, IMG_SIZE],
        },
        "augmentation": {
            "rotation_degrees": 20,
            "translation_fraction": 0.06,
            "horizontal_vertical_flip": True,
            "zoom": 0.10,
            "contrast": 0.10,
        },
        "xai": ["Grad-CAM", "Grad-CAM++", "LIME", "SHAP", "Saliency Map"],
    }
    with open(RESULTS_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(final_metadata, f, indent=2)

    zip_path = zip_results()
    print("\nCompleted successfully.")
    print("Results directory:", RESULTS_DIR)
    print("ZIP file:", zip_path)
    auto_download(zip_path)

if __name__ == "__main__":
    main()


TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">
Kaggle credentials found: /root/.kaggle/kaggle.json
[CMD] kaggle kernels pull ahmedhamza1996/liver-tumor-classification -p /content/liver_efficientnetv2s_work/kaggle_kernel_metadata -m
Discovered Kaggle dataset sources: ['ahmedhamza1996/dataset']
[CMD] kaggle datasets download -d ahmedhamza1996/dataset -p /content/liver_efficientnetv2s_work/dataset/ahmedhamza1996__dataset --unzip
Dataset URL: https://www.kaggle.com/datasets/ahmedhamza1996/dataset
License(s): unknown

  0%|          | 0.00/241M [00:00<?, ?B/s]
  3%|▎         | 7.00M/241M [00:00<00:04, 59.5MB/s]
  7%|▋         | 17.0M/241M [00:00<00:03, 67.6MB/s]
 12%|█▏        | 29.0M/241M [00:00<00:04, 53.1MB/s]
 18%|█▊        | 43.0M/241M [00:00<00:03, 68.0MB/s]
 22%|██▏       | 53.0M/241M [00:00<00:02, 66.6MB/s]
 28%|██▊       | 67.0M/241M [00:00<00:02, 83.1MB/s]
 32%|███▏      | 76.

Epoch 1/20
     29/Unknown 86s 2s/step - accuracy: 0.4359 - loss: 1.6332
Epoch 1: val_accuracy improved from None to 0.67808, saving model to /content/liver_efficientnetv2s_work/results_efficientnetv2s/model/best_efficientnetv2s_liver.keras

Epoch 1: finished saving model to /content/liver_efficientnetv2s_work/results_efficientnetv2s/model/best_efficientnetv2s_liver.keras
 — test_acc: 0.6813 — test_prec: 0.5574 — test_rec/sens: 0.4028 — test_spec: 0.6931 — test_auc: 0.9228
29/29 ━━━━━━━━━━━━━━━━━━━━ 146s 4s/step - accuracy: 0.5276 - loss: 1.2882 - val_accuracy: 0.6781 - val_loss: 0.6777 - learning_rate: 0.0010
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7010 - loss: 0.6913
Epoch 2: val_accuracy did not improve from 0.67808
 — test_acc: 0.6648 — test_prec: 0.5537 — test_rec/sens: 0.3611 — test_spec: 0.6772 — test_auc: 0.9595
29/29 ━━━━━━━━━━━━━━━━━━━━ 92s 3s/step - accuracy: 0.7379 - loss: 0.6555 - val_accuracy: 0.6575 - val_loss: 0.6130 - learning_rate: 0.0010
Epoch 

  0%|          | 0/700 [00:00<?, ?it/s]

Generating XAI 2/20: /content/liver_efficientnetv2s_work/dataset/ahmedhamza1996__dataset/output/benign/(370).PNG


  0%|          | 0/700 [00:00<?, ?it/s]

...

Generating XAI 20/20: /content/liver_efficientnetv2s_work/dataset/ahmedhamza1996__dataset/output/malignant/(55).PNG


  0%|          | 0/700 [00:00<?, ?it/s]


Completed successfully.
Results directory: /content/liver_efficientnetv2s_work/results_efficientnetv2s
ZIP file: /content/liver_efficientnetv2s_work/EfficientNetV2S_Liver_Tumor_Complete_Results.zip
Starting automatic download: /content/liver_efficientnetv2s_work/EfficientNetV2S_Liver_Tumor_Complete_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from google.colab import files

zip_path = "/content/liver_efficientnetv2s_work.zip"

files.download(zip_path)

FileNotFoundError: Cannot find file: /content/liver_efficientnetv2s_work.zip